## Aprendizaje supervisado: Modelo de Regresión

### Contexto y objetivo

En las notebooks anteriores exploramos los datos, construimos una tabla integrada por pozo y mes y preparamos variables predictoras a partir del historial de producción, las características de los pozos y la información de fracturación.

En esta notebook entrenaremos y evaluaremos modelos supervisados para predecir la producción de petróleo del mes siguiente. Comenzaremos con regresión y utilizaremos las particiones temporales preparadas en la etapa anterior.

La evaluación se realizará en orden cronológico: usaremos el conjunto de entrenamiento para ajustar los modelos, el de validación para compararlos y el de test para medir el desempeño final. Antes de entrenar, compararemos los modelos con una referencia sencilla basada en la producción del mes de corte.

Una vez evaluada la regresión, abordaremos la clasificación como un problema separado.

### 1. Carga y verificación de los datasets de modelado

La notebook `03_feature_engineering.ipynb` exportó los conjuntos de entrenamiento, validación, test e inferencia, además de un archivo con la definición de las variables. En esta sección cargamos esos archivos y comprobamos sus dimensiones, períodos y columnas antes de comenzar el modelado.

Los conjuntos de entrenamiento, validación y test contienen el valor real de producción del mes siguiente. El conjunto de inferencia representa los registros para los que queremos generar una predicción y, por ese motivo, no tiene ese valor objetivo.

In [1]:
import json
from pathlib import Path

import pandas as pd

# Buscamos la carpeta del proyecto a partir de la ubicación de la notebook.
ubicacion = Path.cwd()
ROOT = next(
    (
        ruta
        for ruta in [ubicacion, *ubicacion.parents]
        if (ruta / "data" / "processed").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "No encontramos la carpeta data/processed. "
        "Revisá desde qué carpeta se abrió la notebook."
    )

RUTA_PROCESSED = ROOT / "data" / "processed"

# Estos archivos deben haberse generado al ejecutar la notebook 03.
archivos_requeridos = [
    "train_modelado.csv",
    "validation_modelado.csv",
    "test_modelado.csv",
    "inferencia_modelado.csv",
    "columnas_modelado.json",
]

archivos_faltantes = [
    nombre
    for nombre in archivos_requeridos
    if not (RUTA_PROCESSED / nombre).is_file()
]

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan archivos de modelado en data/processed: "
        + ", ".join(archivos_faltantes)
        + ". Revisá que las notebooks 02 y 03 hayan terminado de ejecutarse correctamente."
    )

# Cargamos la definición de predictores, objetivos y particiones.
with open(RUTA_PROCESSED / "columnas_modelado.json", encoding="utf-8") as archivo:
    definicion_modelado = json.load(archivo)

# Cargamos las particiones y reconocemos las columnas de fecha.
fechas = ["fecha", "mes_objetivo"]

train = pd.read_csv(
    RUTA_PROCESSED / "train_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

validation = pd.read_csv(
    RUTA_PROCESSED / "validation_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

test = pd.read_csv(
    RUTA_PROCESSED / "test_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

inferencia = pd.read_csv(
    RUTA_PROCESSED / "inferencia_modelado.csv",
    parse_dates=fechas,
    low_memory=False
)

# Resumimos la cantidad de registros, pozos y períodos de cada conjunto.
resumen_particiones = []

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
    ("Inferencia", inferencia),
]:
    resumen_particiones.append({
        "Conjunto": nombre,
        "Filas": len(datos),
        "Columnas": datos.shape[1],
        "Pozos": datos["idpozo"].nunique(),
        "Desde": datos["fecha"].min().strftime("%Y-%m"),
        "Hasta": datos["fecha"].max().strftime("%Y-%m"),
    })

display(pd.DataFrame(resumen_particiones).set_index("Conjunto"))

# Comprobamos que los predictores definidos estén presentes en las particiones.
features = (
    definicion_modelado["features_numericas"]
    + definicion_modelado["features_booleanas"]
    + definicion_modelado["features_categoricas"]
)

columnas_ausentes = sorted(set(features) - set(train.columns))

if columnas_ausentes:
    raise KeyError(
        "Hay predictores definidos en columnas_modelado.json que no aparecen "
        f"en train_modelado.csv: {columnas_ausentes}"
    )

print(f"\nCantidad de variables predictoras definidas: {len(features)}")
print(f"Objetivo de regresión: {definicion_modelado['target_regresion']}")
print(f"Objetivo de clasificación: {definicion_modelado['target_clasificacion']}")
print("\nPeríodos del objetivo por partición:")
for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
]:
    print(
        f"- {nombre}: "
        f"{datos['mes_objetivo'].min():%Y-%m} a "
        f"{datos['mes_objetivo'].max():%Y-%m}"
    )

,Filas,Columnas,Pozos,Desde,Hasta
Conjunto,,,,,
Entrenamiento,49452,58,4707,2025-01,2025-11
Validación,14320,58,4814,2025-12,2026-02
Test,19737,58,5015,2026-03,2026-06
Inferencia,5062,56,5062,2026-07,2026-07



Cantidad de variables predictoras definidas: 41
Objetivo de regresión: prod_pet_mes_siguiente
Objetivo de clasificación: clase_prod_pet_mes_siguiente

Períodos del objetivo por partición:
- Entrenamiento: 2025-02 a 2025-12
- Validación: 2026-01 a 2026-03
- Test: 2026-04 a 2026-07


### 2. Modelo de referencia para regresión

Antes de entrenar modelos de Machine Learning, establecemos una referencia sencilla para comparar los resultados. Suponemos que la producción de petróleo del mes siguiente será igual a la producción del mes de corte.

Esta predicción se conoce como persistencia. Por ejemplo, si un pozo produjo 80 m³ durante el mes de corte, la referencia estima que producirá 80 m³ el mes siguiente.

Evaluamos esta referencia con el conjunto de validación, que utilizaremos para comparar modelos. Por el momento no calculamos métricas sobre el conjunto de test, ya que lo reservaremos para la evaluación final.

In [2]:
import numpy as np

# Verificamos que todas las particiones tengan las variables predictoras esperadas.
particiones = {
    "Entrenamiento": train,
    "Validación": validation,
    "Test": test,
    "Inferencia": inferencia,
}

for nombre, datos in particiones.items():
    faltantes = sorted(set(features) - set(datos.columns))

    if faltantes:
        raise KeyError(
            f"Faltan variables predictoras en {nombre}: {faltantes}"
        )

# El objetivo debe estar disponible en entrenamiento, validación y test,
# pero no en el conjunto que se utilizará para generar predicciones futuras.
objetivo = definicion_modelado["target_regresion"]

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
]:
    if objetivo not in datos.columns:
        raise KeyError(f"No se encontró el objetivo de regresión en {nombre}.")

    if datos[objetivo].isna().any():
        raise ValueError(f"Hay valores faltantes en el objetivo de {nombre}.")

if objetivo in inferencia.columns:
    raise ValueError(
        "El conjunto de inferencia no debería incluir el objetivo, "
        "porque todavía no conocemos la producción del mes siguiente."
    )

# En la referencia, usamos la producción del mes de corte
# como estimación de la producción del mes siguiente.
y_real_validacion = validation[objetivo]
y_pred_referencia = validation["prod_pet"]

if y_pred_referencia.isna().any():
    raise ValueError(
        "Hay valores faltantes en prod_pet de validación y no se puede "
        "calcular la referencia para todos los registros."
    )

error = y_real_validacion - y_pred_referencia

mae_referencia = np.abs(error).mean()
rmse_referencia = np.sqrt((error ** 2).mean())

print(f"Registros evaluados en validación: {len(validation):,}")
print(f"MAE de la referencia: {mae_referencia:.2f} m³")
print(f"RMSE de la referencia: {rmse_referencia:.2f} m³")
print(
    "Porcentaje de valores reales de producción iguales a cero: "
    f"{(y_real_validacion == 0).mean() * 100:.1f}%"
)

Registros evaluados en validación: 14,320
MAE de la referencia: 160.04 m³
RMSE de la referencia: 506.07 m³
Porcentaje de valores reales de producción iguales a cero: 30.1%


### 3. Distribución del objetivo y análisis de la referencia

La referencia de persistencia predice que la producción del mes siguiente será igual a la producción del mes de corte. Antes de entrenar modelos, analizamos la distribución de la variable objetivo y los errores de esta referencia.

Este análisis nos permitirá interpretar las métricas generales y observar si la referencia se comporta de manera diferente cuando la producción real del mes siguiente es igual a cero o es positiva. Utilizamos el conjunto de validación para este análisis y mantenemos el conjunto de test apartado para la evaluación final.

In [3]:
# Resumimos la distribución de la producción futura en cada partición.
resumen_objetivo = []

for nombre, datos in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Test", test),
]:
    objetivo_datos = datos[objetivo]

    resumen_objetivo.append({
        "Conjunto": nombre,
        "Registros": len(datos),
        "Producción media (m³)": objetivo_datos.mean(),
        "Producción mediana (m³)": objetivo_datos.median(),
        "Percentil 90 (m³)": objetivo_datos.quantile(0.90),
        "Percentil 99 (m³)": objetivo_datos.quantile(0.99),
        "Producción igual a cero (%)": (objetivo_datos == 0).mean() * 100,
    })

resumen_objetivo = pd.DataFrame(resumen_objetivo).set_index("Conjunto")
display(resumen_objetivo.round(2))

# Analizamos por separado los errores de la referencia
# según la producción real del mes siguiente.
analisis_referencia = validation[
    [objetivo, "prod_pet"]
].copy()

analisis_referencia["grupo_produccion_real"] = np.where(
    analisis_referencia[objetivo] == 0,
    "Producción real igual a cero",
    "Producción real positiva"
)

resultados_por_grupo = []

for grupo, datos in analisis_referencia.groupby(
    "grupo_produccion_real",
    observed=True
):
    errores = datos[objetivo] - datos["prod_pet"]

    resultados_por_grupo.append({
        "Grupo": grupo,
        "Registros": len(datos),
        "Porcentaje de validación (%)": len(datos) / len(validation) * 100,
        "MAE de la referencia (m³)": np.abs(errores).mean(),
        "RMSE de la referencia (m³)": np.sqrt((errores ** 2).mean()),
    })

display(pd.DataFrame(resultados_por_grupo).set_index("Grupo").round(2))

,Registros,Producción media (m³),Producción mediana (m³),Percentil 90 (m³),Percentil 99 (m³),Producción igual a cero (%)
Conjunto,,,,,,
Entrenamiento,49452,545.12,12.50,1768.62,6057.91,32.05
Validación,14320,596.48,15.69,1838.86,6743.40,30.13
Test,19737,615.72,18.07,1883.95,6343.12,29.09


,Registros,Porcentaje de validación (%),MAE de la referencia (m³),RMSE de la referencia (m³)
Grupo,,,,
Producción real igual a cero,4315,30.13,11.77,140.66
Producción real positiva,10005,69.87,223.99,598.36


### 4. Preparación de las variables para regresión

Para entrenar un modelo necesitamos separar las variables predictoras del valor que queremos estimar. Utilizaremos las 41 variables definidas en la notebook 03 como entradas (`X`) y `prod_pet_mes_siguiente` como objetivo (`y`).

Cada fila representa un pozo en un mes de corte: las variables de entrada describen la información disponible hasta ese mes y el objetivo contiene la producción del mes calendario siguiente. El objetivo no se incluye entre las variables predictoras.

En esta etapa preparamos las matrices de entrenamiento y validación y revisamos los tipos de datos y los valores faltantes. Todavía no imputamos ni codificamos variables. Esas transformaciones se incorporarán junto con el modelo para que se ajusten únicamente con los datos de entrenamiento. El conjunto de test se mantiene apartado.

In [4]:
# Verificamos que el objetivo no esté incluido entre las variables predictoras.
if objetivo in features:
    raise ValueError(
        f"El objetivo {objetivo} no debe formar parte de las variables predictoras."
    )

# Separamos las entradas (X) y el objetivo (y) para entrenamiento y validación.
X_train = train[features].copy()
y_train = train[objetivo].copy()

X_validation = validation[features].copy()
y_validation = validation[objetivo].copy()

# Revisamos que los objetivos estén completos.
if y_train.isna().any() or y_validation.isna().any():
    raise ValueError("Hay valores faltantes en los objetivos de entrenamiento o validación.")

print(f"Variables predictoras: {len(features)}")
print(f"Filas de entrenamiento: {len(X_train):,}")
print(f"Filas de validación: {len(X_validation):,}")
print(f"Objetivo: {objetivo}")

# Comparamos los tipos de datos y los faltantes de las variables de entrada.
resumen_variables = pd.DataFrame({
    "Tipo de dato": X_train.dtypes.astype(str),
    "% faltantes en entrenamiento": (X_train.isna().mean() * 100).round(2),
    "% faltantes en validación": (X_validation.isna().mean() * 100).round(2),
})

display(
    resumen_variables
    .sort_values("% faltantes en entrenamiento", ascending=False)
)

print("\nCantidad de variables por tipo de dato:")
display(X_train.dtypes.astype(str).value_counts().to_frame("Cantidad"))

Variables predictoras: 41
Filas de entrenamiento: 49,452
Filas de validación: 14,320
Objetivo: prod_pet_mes_siguiente


,Tipo de dato,% faltantes en entrenamiento,% faltantes en validación
prod_pet_cambio_rel,float64,33.13,31.30
potencia_equipos_fractura_hp_max,float64,29.66,26.56
presion_maxima_psi_max,float64,29.37,26.28
eventos_fractura_acum,float64,29.35,26.26
cantidad_fracturas_acum,float64,29.35,26.26
meses_desde_ultima_fractura,float64,29.35,26.26
longitud_rama_horizontal_m_max,float64,29.35,26.26
arena_bombeada_tn_acum,float64,29.35,26.26
agua_inyectada_m3_acum,float64,29.35,26.26
tipo_terminacion_ultima,str,29.35,26.26



Cantidad de variables por tipo de dato:


,Cantidad
float64,26
str,11
int64,2
bool,2


### 5. Primer modelo de regresión

En esta sección entrenamos un primer modelo para estimar `prod_pet_mes_siguiente`. Utilizamos como variables predictoras las columnas definidas en la notebook 03 y ajustamos el modelo únicamente con el conjunto de entrenamiento.

Antes del entrenamiento, incorporamos dentro de un `Pipeline` el tratamiento de valores faltantes y la codificación de las variables categóricas. De esta manera, las transformaciones se ajustan con los datos de entrenamiento y se aplican sin volver a estimarlas sobre validación.

Como primer algoritmo utilizamos `RandomForestRegressor`. Este modelo combina varios árboles de decisión y permite representar relaciones no lineales entre las variables. En esta etapa no buscamos encontrar la configuración óptima: queremos obtener una primera medida de desempeño y compararla con la referencia de persistencia.

Evaluamos el resultado con MAE, RMSE y R² sobre validación. El conjunto de test se mantiene apartado para la evaluación final.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Recuperamos los grupos de variables según su tipo.
features_numericas = definicion_modelado["features_numericas"]
features_booleanas = definicion_modelado["features_booleanas"]
features_categoricas = definicion_modelado["features_categoricas"]

# Las variables booleanas se pueden tratar como entradas numéricas.
columnas_numericas = features_numericas + features_booleanas

# Para los faltantes numéricos usamos medianas aprendidas solo en entrenamiento.
preprocesamiento_numerico = Pipeline([
    ("imputador", SimpleImputer(strategy="median", add_indicator=True))
])

# Los faltantes categóricos se identifican explícitamente.
# OneHotEncoder convierte las categorías en columnas numéricas.
preprocesamiento_categorico = Pipeline([
    ("imputador", SimpleImputer(strategy="constant", fill_value="Sin dato")),
    ("codificador", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

# Aplicamos un preprocesamiento distinto según el tipo de variable.
preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", preprocesamiento_numerico, columnas_numericas),
        ("categoricas", preprocesamiento_categorico, features_categoricas),
    ],
    remainder="drop"
)

# Creamos un primer modelo de bosque aleatorio.
modelo_regresion = Pipeline([
    ("preprocesamiento", preprocesador),
    ("modelo", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Ajustamos el preprocesamiento y el modelo usando solamente entrenamiento.
modelo_regresion.fit(X_train, y_train)

# Generamos predicciones para validación, que no se usó para ajustar el modelo.
y_pred_modelo = modelo_regresion.predict(X_validation)

# Calculamos las métricas de la referencia y del primer modelo.
resultados_modelos = pd.DataFrame({
    "MAE (m³)": [
        mean_absolute_error(y_validation, y_pred_referencia),
        mean_absolute_error(y_validation, y_pred_modelo),
    ],
    "RMSE (m³)": [
        np.sqrt(mean_squared_error(y_validation, y_pred_referencia)),
        np.sqrt(mean_squared_error(y_validation, y_pred_modelo)),
    ],
    "R²": [
        r2_score(y_validation, y_pred_referencia),
        r2_score(y_validation, y_pred_modelo),
    ],
}, index=["Persistencia", "Random Forest"])

display(resultados_modelos.round(3))

,MAE (m³),RMSE (m³),R²
Persistencia,160.039,506.071,0.85
Random Forest,128.364,368.283,0.92


### 6. Análisis de errores en validación

Las métricas generales resumen el desempeño de cada método, pero no muestran en qué tipo de registros se producen los errores. Por eso comparamos la referencia de persistencia y el Random Forest en dos grupos: observaciones cuya producción real del mes siguiente fue igual a cero y observaciones con producción positiva.

Para cada grupo calculamos el MAE y el RMSE. También calculamos el error promedio con signo, definido como predicción menos valor real: un resultado positivo indica que, en promedio, se sobreestima la producción; uno negativo indica que se la subestima.

Este análisis se realiza en validación. El conjunto de test continúa apartado y no se utiliza para elegir ni ajustar modelos.

In [6]:
# Reunimos los valores reales y las predicciones de validación.
analisis_errores = pd.DataFrame({
    "Producción real": y_validation,
    "Persistencia": y_pred_referencia,
    "Random Forest": y_pred_modelo,
})

# Separamos los registros según la producción real del mes siguiente.
analisis_errores["Grupo"] = np.where(
    analisis_errores["Producción real"] == 0,
    "Producción real igual a cero",
    "Producción real positiva"
)

resultados_errores = []

for grupo, datos in analisis_errores.groupby("Grupo", sort=False):
    y_real_grupo = datos["Producción real"]

    for nombre_modelo in ["Persistencia", "Random Forest"]:
        y_pred_grupo = datos[nombre_modelo]
        error = y_pred_grupo - y_real_grupo

        resultados_errores.append({
            "Grupo": grupo,
            "Modelo": nombre_modelo,
            "Registros": len(datos),
            "MAE (m³)": np.abs(error).mean(),
            "RMSE (m³)": np.sqrt((error ** 2).mean()),
            "Error promedio (m³)": error.mean(),
            "Producción real media (m³)": y_real_grupo.mean(),
            "Predicción media (m³)": y_pred_grupo.mean(),
        })

resultados_errores = (
    pd.DataFrame(resultados_errores)
    .set_index(["Grupo", "Modelo"])
)

display(resultados_errores.round(2))

Registros  MAE (m³)  RMSE (m³)  \
Grupo                        Modelo                                          
Producción real positiva     Persistencia       10005    223.99     598.36   
                             Random Forest      10005    174.38     427.87   
Producción real igual a cero Persistencia        4315     11.77     140.66   
                             Random Forest       4315     21.66     160.10   

                                            Error promedio (m³)  \
Grupo                        Modelo                               
Producción real positiva     Persistencia                 -3.93   
                             Random Forest               -14.78   
Producción real igual a cero Persistencia                 11.77   
                             Random Forest                21.66   

                                            Producción real media (m³)  \
Grupo                        Modelo                                      
Producción real positiva     Persistencia                       853.73   
                             Random Forest                      853.73   
Producción real igual a cero Persistencia                         0.00   
                             Random Forest                        0.00   

                                            Predicción media (m³)  
Grupo                        Modelo                                
Producción real positiva     Persistencia                  849.81  
                             Random Forest                 838.95  
Producción real igual a cero Persistencia                   11.77  
                             Random Forest                  21.66

### 7. Comparación con un modelo de regresión lineal

El análisis de errores muestra que Random Forest mejora a la referencia de persistencia en los registros con producción positiva, pero presenta errores mayores en los registros cuya producción real es igual a cero. Para ampliar la comparación, entrenamos un modelo Ridge, que es una regresión lineal regularizada.

Ridge estima la producción a partir de una combinación lineal de las variables predictoras y aplica una penalización para limitar coeficientes demasiado grandes. A diferencia de Random Forest, no construye árboles ni modela automáticamente relaciones no lineales. Por ese motivo, esta comparación nos permitirá evaluar si un modelo más simple puede obtener resultados competitivos.

Las variables numéricas se imputan y escalan, y las categóricas se imputan y codifican. Estas transformaciones se ajustan únicamente con entrenamiento y se aplican a validación. Comparamos ambos modelos y la referencia con las mismas métricas y particiones. El conjunto de test continúa apartado.

In [7]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Preparamos las variables numéricas para Ridge:
# completamos faltantes y escalamos las variables.
preprocesamiento_numerico_ridge = Pipeline([
    ("imputador", SimpleImputer(strategy="median", add_indicator=True)),
    ("escalador", StandardScaler())
])

# Completamos faltantes categóricos y codificamos las categorías.
preprocesamiento_categorico_ridge = Pipeline([
    ("imputador", SimpleImputer(strategy="constant", fill_value="Sin dato")),
    ("codificador", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=20
    ))
])

# Aplicamos el tratamiento correspondiente a cada grupo de variables.
preprocesador_ridge = ColumnTransformer(
    transformers=[
        ("numericas", preprocesamiento_numerico_ridge, columnas_numericas),
        ("categoricas", preprocesamiento_categorico_ridge, features_categoricas),
    ],
    remainder="drop"
)

# Creamos una primera versión de Ridge.
# alpha controla la intensidad de la regularización.
modelo_ridge = Pipeline([
    ("preprocesamiento", preprocesador_ridge),
    ("modelo", Ridge(alpha=10.0))
])

# Ajustamos el preprocesamiento y el modelo solo con los datos de entrenamiento.
modelo_ridge.fit(X_train, y_train)

# Predecimos la producción del mes siguiente para validación.
y_pred_ridge = modelo_ridge.predict(X_validation)

# Comparamos los tres métodos sobre las mismas observaciones de validación.
predicciones_validacion = {
    "Persistencia": y_pred_referencia,
    "Random Forest": y_pred_modelo,
    "Ridge": y_pred_ridge,
}

comparacion_validacion = pd.DataFrame(
    {
        "MAE (m³)": {
            nombre: mean_absolute_error(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
        "RMSE (m³)": {
            nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
            for nombre, prediccion in predicciones_validacion.items()
        },
        "R²": {
            nombre: r2_score(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
    }
)

display(comparacion_validacion.round(3))

# Revisamos si Ridge produce estimaciones negativas.
print(
    "Predicciones negativas de Ridge en validación: "
    f"{(y_pred_ridge < 0).sum():,} "
    f"({(y_pred_ridge < 0).mean() * 100:.2f}%)"
)

,MAE (m³),RMSE (m³),R²
Persistencia,160.039,506.071,0.850
Random Forest,128.364,368.283,0.920
Ridge,166.814,402.679,0.905


Predicciones negativas de Ridge en validación: 4,190 (29.26%)


### 8. Resultados iniciales y revisión de predicciones negativas

En las primeras comparaciones, Random Forest obtuvo menores valores de MAE y RMSE que la referencia de persistencia en el conjunto de validación. Al analizar los errores por separado, observamos que mejora las predicciones cuando la producción real es positiva, pero presenta errores mayores cuando la producción real es igual a cero.

Ridge obtuvo un MAE algo mayor que la referencia y generó predicciones negativas, aunque la producción de petróleo no puede ser negativa. Para evaluar el efecto de esta limitación, calculamos una variante en la que reemplazamos por cero las predicciones negativas de Ridge. Comparamos sus métricas con las de los métodos anteriores, usando las mismas observaciones de validación.

Esta corrección se evalúa como un experimento explícito: no cambia los valores reales ni el entrenamiento de Ridge. El conjunto de test se mantiene apartado.

In [8]:
# Aplicamos una restricción física a las predicciones de Ridge:
# la producción estimada no puede ser menor que cero.
y_pred_ridge_no_negativa = np.maximum(y_pred_ridge, 0)

# Comparamos los métodos sobre las mismas observaciones de validación.
predicciones_comparacion = {
    "Persistencia": y_pred_referencia,
    "Random Forest": y_pred_modelo,
    "Ridge": y_pred_ridge,
    "Ridge con mínimo cero": y_pred_ridge_no_negativa,
}

resultados_comparacion = pd.DataFrame(
    {
        "MAE (m³)": {
            nombre: mean_absolute_error(y_validation, prediccion)
            for nombre, prediccion in predicciones_comparacion.items()
        },
        "RMSE (m³)": {
            nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
            for nombre, prediccion in predicciones_comparacion.items()
        },
        "R²": {
            nombre: r2_score(y_validation, prediccion)
            for nombre, prediccion in predicciones_comparacion.items()
        },
        "Predicciones negativas": {
            nombre: (prediccion < 0).sum()
            for nombre, prediccion in predicciones_comparacion.items()
        },
    }
)

display(resultados_comparacion.round(3))

,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,160.039,506.071,0.850,0
Random Forest,128.364,368.283,0.920,0
Ridge,166.814,402.679,0.905,4190
Ridge con mínimo cero,154.138,401.138,0.905,0


### 9. Validación temporal del primer modelo

En la comparación inicial, Random Forest obtuvo mejores métricas que la referencia de persistencia y Ridge en el conjunto de validación. Para comprobar si su desempeño se mantiene en distintos períodos, realizamos una validación temporal dentro del conjunto de entrenamiento.

En cada pliegue, entrenamos con los meses del objetivo anteriores y evaluamos en uno o más meses posteriores. No mezclamos los períodos de manera aleatoria, ya que queremos simular el uso del modelo para predecir el futuro.

Comparamos Random Forest con la referencia de persistencia utilizando MAE y RMSE. Esta validación sirve para estudiar la estabilidad del modelo y no reemplaza la evaluación final en test, que continúa apartado.

In [9]:
from sklearn.base import clone

# Definimos pliegues cronológicos usando el mes del objetivo.
# En cada pliegue, los meses de validación son posteriores a los de entrenamiento.
pliegues_temporales = [
    {
        "Pliegue": "1",
        "Entrenamiento hasta": "2025-07-01",
        "Validación desde": "2025-08-01",
        "Validación hasta": "2025-09-01",
    },
    {
        "Pliegue": "2",
        "Entrenamiento hasta": "2025-09-01",
        "Validación desde": "2025-10-01",
        "Validación hasta": "2025-11-01",
    },
    {
        "Pliegue": "3",
        "Entrenamiento hasta": "2025-11-01",
        "Validación desde": "2025-12-01",
        "Validación hasta": "2025-12-01",
    },
]

resultados_pliegues = []
objetivos_reales_pliegues = []
predicciones_rf_pliegues = []
predicciones_persistencia_pliegues = []

for pliegue in pliegues_temporales:
    fin_entrenamiento = pd.Timestamp(pliegue["Entrenamiento hasta"])
    inicio_validacion = pd.Timestamp(pliegue["Validación desde"])
    fin_validacion = pd.Timestamp(pliegue["Validación hasta"])

    mascara_entrenamiento = train["mes_objetivo"] <= fin_entrenamiento
    mascara_validacion = train["mes_objetivo"].between(
        inicio_validacion,
        fin_validacion
    )

    datos_entrenamiento = train.loc[mascara_entrenamiento]
    datos_validacion = train.loc[mascara_validacion]

    if datos_entrenamiento.empty or datos_validacion.empty:
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no tiene filas suficientes."
        )

    # Confirmamos que validemos siempre en meses posteriores al entrenamiento.
    if datos_entrenamiento["mes_objetivo"].max() >= datos_validacion["mes_objetivo"].min():
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no respeta el orden temporal."
        )

    # Creamos una copia sin ajustar del pipeline y la entrenamos solo en este pliegue.
    modelo_pliegue = clone(modelo_regresion)
    modelo_pliegue.fit(
        datos_entrenamiento[features],
        datos_entrenamiento[objetivo]
    )

    prediccion_rf = modelo_pliegue.predict(datos_validacion[features])
    prediccion_persistencia = datos_validacion["prod_pet"].to_numpy()
    y_real = datos_validacion[objetivo].to_numpy()

    resultados_pliegues.append({
        "Pliegue": pliegue["Pliegue"],
        "Objetivos de validación": (
            f"{datos_validacion['mes_objetivo'].min():%Y-%m} a "
            f"{datos_validacion['mes_objetivo'].max():%Y-%m}"
        ),
        "Filas de entrenamiento": len(datos_entrenamiento),
        "Filas de validación": len(datos_validacion),
        "MAE persistencia (m³)": mean_absolute_error(y_real, prediccion_persistencia),
        "MAE Random Forest (m³)": mean_absolute_error(y_real, prediccion_rf),
        "RMSE persistencia (m³)": np.sqrt(
            mean_squared_error(y_real, prediccion_persistencia)
        ),
        "RMSE Random Forest (m³)": np.sqrt(
            mean_squared_error(y_real, prediccion_rf)
        ),
    })

    objetivos_reales_pliegues.extend(y_real)
    predicciones_rf_pliegues.extend(prediccion_rf)
    predicciones_persistencia_pliegues.extend(prediccion_persistencia)

display(pd.DataFrame(resultados_pliegues).set_index("Pliegue").round(2))

# Calculamos también las métricas agrupadas de todas las predicciones temporales.
y_real_pliegues = np.asarray(objetivos_reales_pliegues)
y_rf_pliegues = np.asarray(predicciones_rf_pliegues)
y_persistencia_pliegues = np.asarray(predicciones_persistencia_pliegues)

resumen_validacion_temporal = pd.DataFrame(
    {
        "MAE (m³)": [
            mean_absolute_error(y_real_pliegues, y_persistencia_pliegues),
            mean_absolute_error(y_real_pliegues, y_rf_pliegues),
        ],
        "RMSE (m³)": [
            np.sqrt(mean_squared_error(y_real_pliegues, y_persistencia_pliegues)),
            np.sqrt(mean_squared_error(y_real_pliegues, y_rf_pliegues)),
        ],
    },
    index=["Persistencia", "Random Forest"]
)

print("\nMétricas agrupadas de los pliegues temporales:")
display(resumen_validacion_temporal.round(2))


# Convertimos los pliegues a pares de posiciones dentro de X_train.
# Este formato permite usarlos como parámetro cv de GridSearchCV.
mes_objetivo_train = train["mes_objetivo"].reset_index(drop=True)
pliegues_cv = []

for pliegue in pliegues_temporales:
    mascara_entrenamiento = mes_objetivo_train <= pd.Timestamp(pliegue["Entrenamiento hasta"])
    mascara_validacion = mes_objetivo_train.between(
        pd.Timestamp(pliegue["Validación desde"]),
        pd.Timestamp(pliegue["Validación hasta"]),
    )
    pliegues_cv.append((
        np.flatnonzero(mascara_entrenamiento.to_numpy()),
        np.flatnonzero(mascara_validacion.to_numpy()),
    ))

,Objetivos de validación,Filas de entrenamiento,Filas de validación,MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Pliegue,,,,,,,
1,2025-08 a 2025-09,26323,9118,161.31,132.31,522.44,398.24
2,2025-10 a 2025-11,35441,9307,166.55,128.46,559.03,388.83
3,2025-12 a 2025-12,44748,4704,163.76,135.94,562.44,411.59



Métricas agrupadas de los pliegues temporales:


,MAE (m³),RMSE (m³)
Persistencia,163.92,545.62
Random Forest,131.50,397.26


#### 9.1. Búsqueda del `alpha` de Ridge con `GridSearchCV`

En la sección 7 fijamos manualmente `alpha=10` para Ridge. Ahora buscamos ese valor de forma sistemática con `GridSearchCV`, usando los mismos tres pliegues temporales.

No usamos `TimeSeriesSplit` directamente porque divide las filas por posición y, como cada mes contiene miles de pozos, podría separar un mismo mes entre entrenamiento y validación. Los pliegues se definen por **mes del objetivo**, con el mismo esquema de `TimeSeriesSplit`: entrenamiento creciente y validación siempre posterior. Se los pasamos a `GridSearchCV` como una lista de índices.

También descartamos una validación agrupada por pozo (`GroupKFold`), porque mezclaría meses y permitiría entrenar con objetivos posteriores a los evaluados.

Como la producción no puede ser negativa, evaluamos cada `alpha` con el **MAE de las predicciones recortadas en cero**, la misma restricción física que aplicamos en la sección 8. De esta manera la búsqueda elige el `alpha` que mejor funciona con esa restricción.

In [10]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV


def mae_no_negativo(y_real, y_predicho):
    # MAE luego de reemplazar por cero las predicciones negativas.
    return mean_absolute_error(y_real, np.maximum(y_predicho, 0))


puntuacion_mae_no_negativo = make_scorer(mae_no_negativo, greater_is_better=False)

busqueda_ridge = GridSearchCV(
    estimator=clone(modelo_ridge),
    param_grid={"modelo__alpha": [0.1, 1, 10, 100, 1000, 10000]},
    scoring={
        "MAE no negativo": puntuacion_mae_no_negativo,
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
    },
    refit="MAE no negativo",
    cv=pliegues_cv,
    n_jobs=-1,
)

busqueda_ridge.fit(X_train, y_train)

resultados_ridge = pd.DataFrame(busqueda_ridge.cv_results_)
tabla_ridge = pd.DataFrame({
    "alpha": resultados_ridge["param_modelo__alpha"],
    "MAE no negativo medio (m³)": -resultados_ridge["mean_test_MAE no negativo"],
    "MAE no negativo desvío (m³)": resultados_ridge["std_test_MAE no negativo"],
    "MAE medio (m³)": -resultados_ridge["mean_test_MAE"],
    "RMSE medio (m³)": -resultados_ridge["mean_test_RMSE"],
}).set_index("alpha")

display(tabla_ridge.round(2))

mejor_alpha = busqueda_ridge.best_params_["modelo__alpha"]
print(f"Mejor alpha según los pliegues temporales: {mejor_alpha}")

# Comparamos en validación el alpha manual con el elegido por la búsqueda.
y_pred_ridge_busqueda = np.maximum(busqueda_ridge.best_estimator_.predict(X_validation), 0)

comparacion_ridge = pd.DataFrame({
    "MAE (m³)": {
        "Ridge alpha=10 (manual), mínimo cero": mean_absolute_error(y_validation, y_pred_ridge_no_negativa),
        f"Ridge alpha={mejor_alpha} (GridSearchCV), mínimo cero": mean_absolute_error(y_validation, y_pred_ridge_busqueda),
        "Random Forest inicial": mean_absolute_error(y_validation, y_pred_modelo),
        "Persistencia": mean_absolute_error(y_validation, y_pred_referencia),
    },
    "RMSE (m³)": {
        "Ridge alpha=10 (manual), mínimo cero": np.sqrt(mean_squared_error(y_validation, y_pred_ridge_no_negativa)),
        f"Ridge alpha={mejor_alpha} (GridSearchCV), mínimo cero": np.sqrt(mean_squared_error(y_validation, y_pred_ridge_busqueda)),
        "Random Forest inicial": np.sqrt(mean_squared_error(y_validation, y_pred_modelo)),
        "Persistencia": np.sqrt(mean_squared_error(y_validation, y_pred_referencia)),
    },
})

print("Comparación en validación (enero a marzo de 2026):")
display(comparacion_ridge.round(2))

,MAE no negativo medio (m³),MAE no negativo desvío (m³),MAE medio (m³),RMSE medio (m³)
alpha,,,,
0.1,165.04,1.67,170.32,446.91
1.0,164.93,1.73,170.16,446.87
10.0,164.71,1.92,169.77,446.73
100.0,164.37,2.60,169.02,447.36
1000.0,163.37,3.15,167.43,444.90
10000.0,172.70,4.58,176.88,460.66


Mejor alpha según los pliegues temporales: 1000
Comparación en validación (enero a marzo de 2026):


,MAE (m³),RMSE (m³)
"Ridge alpha=10 (manual), mínimo cero",154.14,401.14
"Ridge alpha=1000 (GridSearchCV), mínimo cero",151.87,406.10
Random Forest inicial,128.36,368.28
Persistencia,160.04,506.07


La búsqueda eligió `alpha=1000`, una regularización más fuerte que la fijada manualmente (`alpha=10`). En los pliegues temporales, el MAE con mínimo cero bajó de 164,71 m³ a 163,37 m³, y en validación de 154,14 m³ a 151,87 m³. Con `alpha=10000` el error vuelve a aumentar, por lo que el valor elegido se ubica en un punto intermedio y no en el extremo de la grilla.

Aun con el mejor `alpha`, Ridge queda lejos de Random Forest (MAE de 128,36 m³ en validación). Esto confirma que la relación entre las variables y la producción futura no es lineal, y que la diferencia entre ambos modelos no se debía a una mala elección de la regularización. Por eso continuamos con Random Forest como modelo principal.

### 10. Búsqueda de hiperparámetros de Random Forest con `GridSearchCV`

En los tres pliegues temporales, Random Forest obtuvo menores valores de MAE y RMSE que la referencia de persistencia. Esta consistencia es una señal favorable, aunque los pliegues corresponden a meses de 2025 y no garantizan que el desempeño se mantenga en todos los períodos futuros.

En lugar de fijar manualmente los hiperparámetros, los buscamos con `GridSearchCV` sobre los mismos pliegues temporales:

- **`max_depth`**: profundidad máxima de cada árbol. `None` permite árboles sin límite; valores menores generan árboles más simples.
- **`min_samples_leaf`**: cantidad mínima de registros por hoja.
- **`max_features`**: proporción de variables consideradas en cada división.

El preprocesamiento forma parte del pipeline, por lo que en cada pliegue se ajusta solamente con los meses de entrenamiento. Elegimos la configuración con el menor **MAE promedio** entre pliegues. El conjunto de test continúa apartado.

In [11]:
import time

grilla_rf = {
    "modelo__max_depth": [None, 30, 15, 8],
    "modelo__min_samples_leaf": [5, 10],
    "modelo__max_features": [1.0, 0.7],
}

busqueda_rf = GridSearchCV(
    estimator=clone(modelo_regresion),
    param_grid=grilla_rf,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
    },
    refit="MAE",
    cv=pliegues_cv,
    # El Random Forest ya usa todos los núcleos (n_jobs=-1).
    n_jobs=1,
)

cantidad_combinaciones = int(np.prod([len(valores) for valores in grilla_rf.values()]))
print(
    f"Evaluando {cantidad_combinaciones} combinaciones en {len(pliegues_cv)} pliegues "
    f"({cantidad_combinaciones * len(pliegues_cv)} entrenamientos)...",
    flush=True,
)

inicio_busqueda = time.time()
busqueda_rf.fit(X_train, y_train)
print(f"Búsqueda finalizada en {(time.time() - inicio_busqueda) / 60:.1f} minutos.")

resultados_rf = pd.DataFrame(busqueda_rf.cv_results_)

tabla_ajuste_rf = pd.DataFrame({
    "max_depth": resultados_rf["param_modelo__max_depth"].astype(str),
    "min_samples_leaf": resultados_rf["param_modelo__min_samples_leaf"],
    "max_features": resultados_rf["param_modelo__max_features"],
    "MAE medio (m³)": -resultados_rf["mean_test_MAE"],
    "MAE desvío (m³)": resultados_rf["std_test_MAE"],
    "RMSE medio (m³)": -resultados_rf["mean_test_RMSE"],
    "Ranking MAE": resultados_rf["rank_test_MAE"],
}).sort_values("Ranking MAE").reset_index(drop=True)

display(tabla_ajuste_rf.round(2))

mejores_parametros_rf = busqueda_rf.best_params_
print("Mejor configuración:", mejores_parametros_rf)

# Desempeño de la mejor configuración en cada pliegue, junto a la persistencia.
indice_mejor = busqueda_rf.best_index_
tabla_mejor_por_pliegue = pd.DataFrame([
    {
        "Pliegue": pliegue["Pliegue"],
        "MAE persistencia (m³)": mean_absolute_error(
            y_train.iloc[indices_validacion], train["prod_pet"].iloc[indices_validacion]
        ),
        "MAE Random Forest (m³)": -resultados_rf.loc[indice_mejor, f"split{numero}_test_MAE"],
        "RMSE persistencia (m³)": np.sqrt(mean_squared_error(
            y_train.iloc[indices_validacion], train["prod_pet"].iloc[indices_validacion]
        )),
        "RMSE Random Forest (m³)": -resultados_rf.loc[indice_mejor, f"split{numero}_test_RMSE"],
    }
    for numero, (pliegue, (_, indices_validacion)) in enumerate(zip(pliegues_temporales, pliegues_cv))
]).set_index("Pliegue")

print("Mejor configuración por pliegue:")
display(tabla_mejor_por_pliegue.round(2))
print("Promedio y desvío entre pliegues:")
display(tabla_mejor_por_pliegue.agg(["mean", "std"]).round(2))

Evaluando 16 combinaciones en 3 pliegues (48 entrenamientos)...


Búsqueda finalizada en 15.3 minutos.


,max_depth,min_samples_leaf,max_features,MAE medio (m³),MAE desvío (m³),RMSE medio (m³),Ranking MAE
0,NaN,5,0.7,131.39,2.86,395.93,1
1,15,5,0.7,131.44,3.23,396.38,2
2,30,5,0.7,131.64,2.79,396.80,3
3,30,5,1.0,132.22,3.04,399.40,4
4,NaN,5,1.0,132.24,3.05,399.55,5
5,15,5,1.0,132.24,3.32,399.42,6
6,NaN,10,0.7,132.48,3.16,400.10,7
7,30,10,0.7,132.62,3.38,400.18,8
8,15,10,0.7,132.75,3.38,400.42,9
9,8,5,0.7,133.87,4.04,398.35,10


Mejor configuración: {'modelo__max_depth': None, 'modelo__max_features': 0.7, 'modelo__min_samples_leaf': 5}
Mejor configuración por pliegue:


,MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Pliegue,,,,
1,161.31,130.68,522.44,391.88
2,166.55,128.30,559.03,388.84
3,163.76,135.19,562.44,407.07


Promedio y desvío entre pliegues:


,MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
mean,163.87,131.39,547.97,395.93
std,2.62,3.50,22.18,9.76


La búsqueda evaluó las 16 combinaciones en los tres pliegues temporales (48 entrenamientos), con el mismo preprocesamiento.

- **Mejor configuración:** `max_depth=None`, `min_samples_leaf=5` y `max_features=0.7`, con un MAE promedio de 131,39 m³. Es la misma configuración que habíamos elegido en la primera versión del análisis con una comparación manual de tres variantes; ahora queda respaldada por una búsqueda sistemática.
- **La profundidad no conviene limitarla:** con `max_depth=8` se obtuvieron los peores resultados (MAE entre 133,9 y 136,0 m³), mientras que con 15, 30 o sin límite los resultados fueron casi iguales. Los árboles necesitan suficiente profundidad para representar las diferencias entre pozos.
- **`max_features=0.7` fue mejor que 1.0** en todas las profundidades, y `min_samples_leaf=5` fue mejor que 10.
- **Las diferencias entre las mejores configuraciones son pequeñas** (menos de 1 m³ entre las tres primeras) en relación con la variación entre pliegues (desvío de 3,5 m³).

En cada uno de los tres pliegues, la mejor configuración superó a la persistencia: el MAE promedio fue de 131,39 m³ (desvío 3,50) contra 163,87 m³ (desvío 2,62) de la persistencia, y el RMSE de 395,93 m³ contra 547,97 m³. La evaluación ya no depende de una única ventana: la mejora se mantiene en los tres períodos.

Contrastamos esta configuración en validación en la siguiente sección. La comparación se realizó únicamente con pliegues de entrenamiento; el conjunto de test sigue reservado.

### 11. Evaluación de la configuración candidata en validación

En esta sección tomamos la configuración de Random Forest elegida por `GridSearchCV`, la ajustamos con todo el conjunto de entrenamiento y la evaluamos en validación, que contiene meses posteriores. Comparamos sus resultados con la referencia de persistencia y con el Random Forest inicial, utilizando las mismas observaciones y métricas.

Usamos esta comparación para decidir qué configuración llevar a la evaluación final. El conjunto de test se mantiene apartado y no se utiliza para ajustar el modelo.

In [12]:
from sklearn.base import clone

# Creamos la variante candidata a partir del pipeline inicial,
# con los hiperparámetros elegidos por GridSearchCV.
modelo_rf_candidato = clone(modelo_regresion)
modelo_rf_candidato.set_params(**mejores_parametros_rf)
print("Configuración candidata:", mejores_parametros_rf)

# Ajustamos el pipeline completo usando únicamente el conjunto de entrenamiento.
modelo_rf_candidato.fit(X_train, y_train)

# Generamos predicciones para los meses posteriores de validación.
y_pred_rf_candidato = modelo_rf_candidato.predict(X_validation)

# Comparamos todos los métodos sobre las mismas filas de validación.
predicciones_validacion = {
    "Persistencia": y_pred_referencia,
    "Random Forest inicial": y_pred_modelo,
    "Random Forest candidato": y_pred_rf_candidato,
}

comparacion_candidata = pd.DataFrame(
    {
        "MAE (m³)": {
            nombre: mean_absolute_error(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
        "RMSE (m³)": {
            nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
            for nombre, prediccion in predicciones_validacion.items()
        },
        "R²": {
            nombre: r2_score(y_validation, prediccion)
            for nombre, prediccion in predicciones_validacion.items()
        },
        "Predicciones negativas": {
            nombre: (prediccion < 0).sum()
            for nombre, prediccion in predicciones_validacion.items()
        },
    }
)

display(comparacion_candidata.round(3))

Configuración candidata: {'modelo__max_depth': None, 'modelo__max_features': 0.7, 'modelo__min_samples_leaf': 5}


,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,160.039,506.071,0.850,0
Random Forest inicial,128.364,368.283,0.920,0
Random Forest candidato,127.156,364.622,0.922,0


En el conjunto de validación comparamos el Random Forest candidato, con los hiperparámetros elegidos por `GridSearchCV`, contra la persistencia y el Random Forest inicial. Ninguno de los modelos de árboles produce estimaciones negativas.

A partir de esta comparación seleccionamos el Random Forest candidato para la evaluación final. Mantenemos sus parámetros sin cambios y no utilizaremos el conjunto de test para elegir configuraciones.

### 12. Evaluación inicial en test del modelo entrenado desde 2025

Una vez seleccionado el Random Forest candidato con los resultados de validación, medimos su desempeño en el conjunto de test. Este conjunto contiene meses posteriores a los usados para entrenar y validar, por lo que permite evaluar cómo se comporta el modelo ante períodos futuros.

Esta evaluación corresponde a una configuración preliminar del modelo, entrenada con la ventana original desde 2025. 

Comparamos el modelo con la referencia de persistencia usando MAE, RMSE y R². También revisamos el MAE y el RMSE por separado según la producción real del mes siguiente sea cero o positiva, para observar si el desempeño general oculta diferencias entre ambos grupos.

El conjunto de test se utiliza únicamente para esta evaluación final. No modificaremos el modelo ni sus parámetros en función de estos resultados.

In [13]:
# Preparamos la producción real del conjunto de test.
y_test = test[objetivo]

if y_test.isna().any():
    raise ValueError("Hay valores faltantes en el objetivo del conjunto de test.")

# La persistencia predice que el próximo mes tendrá la misma producción
# que el mes de corte.
y_pred_persistencia_test = test["prod_pet"].to_numpy()

# El Random Forest candidato ya fue ajustado con entrenamiento.
# Lo usamos para predecir test sin volver a entrenarlo ni ajustarlo.
X_test = test[features]
y_pred_rf_test = modelo_rf_candidato.predict(X_test)

# Métricas generales de test.
resultados_test = pd.DataFrame(
    {
        "MAE (m³)": [
            mean_absolute_error(y_test, y_pred_persistencia_test),
            mean_absolute_error(y_test, y_pred_rf_test),
        ],
        "RMSE (m³)": [
            np.sqrt(mean_squared_error(y_test, y_pred_persistencia_test)),
            np.sqrt(mean_squared_error(y_test, y_pred_rf_test)),
        ],
        "R²": [
            r2_score(y_test, y_pred_persistencia_test),
            r2_score(y_test, y_pred_rf_test),
        ],
        "Predicciones negativas": [
            (y_pred_persistencia_test < 0).sum(),
            (y_pred_rf_test < 0).sum(),
        ],
    },
    index=["Persistencia", "Random Forest candidato"],
)

display(resultados_test.round(3))

# Analizamos además los errores del modelo elegido por grupo de producción real.
analisis_test = pd.DataFrame({
    "Producción real": y_test.to_numpy(),
    "Predicción Random Forest": y_pred_rf_test,
})

analisis_test["Grupo"] = np.where(
    analisis_test["Producción real"] == 0,
    "Producción real igual a cero",
    "Producción real positiva"
)

resultados_test_por_grupo = []

for grupo, datos in analisis_test.groupby("Grupo", sort=False):
    errores = datos["Predicción Random Forest"] - datos["Producción real"]

    resultados_test_por_grupo.append({
        "Grupo": grupo,
        "Registros": len(datos),
        "Porcentaje de test (%)": len(datos) / len(analisis_test) * 100,
        "MAE (m³)": np.abs(errores).mean(),
        "RMSE (m³)": np.sqrt((errores ** 2).mean()),
        "Error promedio (m³)": errores.mean(),
        "Predicción media (m³)": datos["Predicción Random Forest"].mean(),
    })

display(
    pd.DataFrame(resultados_test_por_grupo)
    .set_index("Grupo")
    .round(2)
)

,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,166.195,551.710,0.820,0
Random Forest candidato,137.144,408.029,0.902,0


,Registros,Porcentaje de test (%),MAE (m³),RMSE (m³),Error promedio (m³),Predicción media (m³)
Grupo,,,,,,
Producción real positiva,13996,70.91,184.38,475.17,-8.53,859.75
Producción real igual a cero,5741,29.09,21.98,148.04,21.98,21.98


#### Resultados de la evaluación final

En el conjunto de test, el Random Forest candidato obtuvo un MAE de 137,14 m³ y un RMSE de 408,03 m³, frente a 166,20 m³ y 551,71 m³ de la referencia de persistencia. En este período, el modelo redujo el MAE aproximadamente un 17 % y el RMSE aproximadamente un 26 % respecto de la referencia.

El análisis por grupo muestra que el error es distinto según la producción real del mes siguiente. En los registros con producción positiva, el MAE fue de 184,38 m³. En los registros con producción igual a cero, el modelo predijo en promedio 21,98 m³, por lo que tendió a sobreestimar esos casos.

Los resultados de test respaldan el uso del Random Forest candidato frente a la referencia para los meses evaluados, aunque sus errores siguen siendo relevantes y su desempeño es menos preciso en los casos con producción real igual a cero. Consideramos realizada la evaluación final y no utilizaremos test para ajustar nuevos parámetros.

### 13. Diagnóstico del modelo por período y tipo de pozo

El Random Forest candidato mejoró las métricas generales respecto de la referencia, pero el resultado promedio no muestra si el desempeño se mantiene en todos los meses y tipos de pozo.

En esta sección analizamos las predicciones de validación por mes del objetivo y por tipo de pozo. Para cada grupo calculamos MAE y RMSE del Random Forest candidato y de la referencia de persistencia. Este análisis es descriptivo: nos permitirá identificar grupos donde los errores son mayores y orientar los próximos experimentos, sin utilizar el conjunto de test para seleccionar modelos.

In [14]:
# Reunimos los valores reales y las predicciones ya calculadas para validación.
diagnostico_validacion = validation[
    ["mes_objetivo", "tipopozo", objetivo, "prod_pet"]
].copy()

diagnostico_validacion["Predicción persistencia"] = (
    y_pred_referencia
)
diagnostico_validacion["Predicción Random Forest"] = (
    y_pred_rf_candidato
)

diagnostico_validacion["Tipo de pozo"] = (
    diagnostico_validacion["tipopozo"]
    .fillna("Sin dato")
    .astype(str)
)

# Resumimos el desempeño por mes del objetivo.
resumen_por_mes = []

for mes, datos in diagnostico_validacion.groupby("mes_objetivo", sort=True):
    y_real = datos[objetivo]

    resumen_por_mes.append({
        "Mes objetivo": mes.strftime("%Y-%m"),
        "Registros": len(datos),
        "Producción real igual a cero (%)": (y_real == 0).mean() * 100,
        "MAE persistencia (m³)": mean_absolute_error(
            y_real, datos["Predicción persistencia"]
        ),
        "MAE Random Forest (m³)": mean_absolute_error(
            y_real, datos["Predicción Random Forest"]
        ),
        "RMSE persistencia (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción persistencia"]
        )),
        "RMSE Random Forest (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción Random Forest"]
        )),
    })

print("Desempeño por mes objetivo en validación:")
display(pd.DataFrame(resumen_por_mes).set_index("Mes objetivo").round(2))

# Resumimos el desempeño por tipo de pozo.
resumen_por_tipo = []

for tipo, datos in diagnostico_validacion.groupby("Tipo de pozo", sort=True):
    y_real = datos[objetivo]

    resumen_por_tipo.append({
        "Tipo de pozo": tipo,
        "Registros": len(datos),
        "Producción real igual a cero (%)": (y_real == 0).mean() * 100,
        "MAE persistencia (m³)": mean_absolute_error(
            y_real, datos["Predicción persistencia"]
        ),
        "MAE Random Forest (m³)": mean_absolute_error(
            y_real, datos["Predicción Random Forest"]
        ),
        "RMSE persistencia (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción persistencia"]
        )),
        "RMSE Random Forest (m³)": np.sqrt(mean_squared_error(
            y_real, datos["Predicción Random Forest"]
        )),
    })

print("\nDesempeño por tipo de pozo en validación:")
display(
    pd.DataFrame(resumen_por_tipo)
    .set_index("Tipo de pozo")
    .sort_values("Registros", ascending=False)
    .round(2)
)

Desempeño por mes objetivo en validación:


,Registros,Producción real igual a cero (%),MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Mes objetivo,,,,,,
2026-01,4736,30.26,148.59,122.86,501.03,373.64
2026-02,4772,30.30,175.57,118.98,500.81,337.27
2026-03,4812,29.84,155.90,139.48,516.10,381.36



Desempeño por tipo de pozo en validación:


,Registros,Producción real igual a cero (%),MAE persistencia (m³),MAE Random Forest (m³),RMSE persistencia (m³),RMSE Random Forest (m³)
Tipo de pozo,,,,,,
Petrolífero,6760,9.59,301.40,233.74,703.47,498.90
Gasífero,6554,40.89,38.43,34.61,221.41,178.25
Otro tipo,922,100.00,0.00,2.00,0.00,13.46
Sumidero,62,100.00,0.00,39.59,0.00,124.93
Inyección de Gas,14,21.43,4.11,4.54,7.93,8.65
Sin dato,8,0.00,294.06,1200.97,317.76,1217.76


En validación, el Random Forest candidato obtuvo menores valores de MAE y RMSE que la persistencia en los tres meses objetivo analizados. La proporción de registros con producción real igual a cero se mantuvo cercana al 30 % en cada mes.

Al separar por tipo de pozo, observamos que Random Forest mejora las métricas en los grupos mayoritarios de pozos petrolíferos y gasíferos. En cambio, presenta errores mayores que la persistencia en algunos grupos con pocas observaciones y producción real igual a cero. Estos grupos pequeños deben interpretarse con cautela, ya que unos pocos errores pueden modificar considerablemente sus métricas.

Por lo tanto, el resultado general es favorable, pero el desempeño no es igual para todos los tipos de pozo. Este diagnóstico orienta los próximos experimentos; no se utilizó el conjunto de test ni se modificó el modelo.

### 14. Estrategia de modelado en dos etapas

En los resultados anteriores observamos que una parte importante de los registros tiene producción de petróleo igual a cero. También vimos que el modelo de regresión puede estimar una producción positiva para algunos casos cuya producción real termina siendo cero.

Para analizar este comportamiento, probaremos una estrategia de modelado en dos etapas. Primero, un clasificador estimará si la producción de petróleo del mes siguiente será cero o positiva. Luego, un modelo de regresión estimará la cantidad producida para los casos con producción positiva.

Las dos etapas se combinan para obtener una predicción final en metros cúbicos. Evaluaremos esa predicción en el conjunto de validación y la compararemos con la referencia de persistencia y con el Random Forest de regresión que ya probamos. El conjunto de test se mantiene apartado.

Esta estrategia no garantiza por sí sola un mejor resultado. La probaremos para comprobar si distinguir primero entre producción nula y positiva ayuda a reducir los errores, en particular las predicciones positivas para casos cuya producción real es cero.

#### 14.1. Primer componente: clasificación de producción positiva

Comenzamos creando una variable binaria a partir de `prod_pet_mes_siguiente`: toma el valor 1 cuando la producción del mes siguiente es positiva y 0 cuando es igual a cero.

Entrenamos un clasificador Random Forest con todos los registros de entrenamiento, ya que necesitamos ejemplos de ambos casos. El preprocesamiento se ajusta solo con entrenamiento y luego se aplica a validación. En esta etapa revisamos la matriz de confusión y métricas de clasificación para conocer la capacidad del modelo de distinguir las dos clases.

Además, guardamos la probabilidad estimada de producción positiva. Más adelante la utilizaremos, junto con las estimaciones del modelo de regresión, para construir y comparar distintas formas de obtener la predicción final en metros cúbicos.

Las métricas de clasificación son diagnósticas: el objetivo principal del proyecto sigue siendo estimar la cantidad de petróleo producida el mes siguiente.

In [15]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

# Creamos una etiqueta binaria: 1 si la producción futura es positiva, 0 si es cero.
y_train_positivo = (y_train > 0).astype(int)
y_validation_positivo = (y_validation > 0).astype(int)

print("Distribución de las clases en entrenamiento:")
display(
    y_train_positivo
    .value_counts()
    .rename(index={0: "Producción igual a cero", 1: "Producción positiva"})
    .to_frame("Registros")
)

# Reutilizamos el preprocesamiento del Random Forest de regresión.
# El pipeline lo ajustará de nuevo usando únicamente el entrenamiento.
clasificador_produccion = Pipeline([
    (
        "preprocesamiento",
        clone(modelo_regresion.named_steps["preprocesamiento"])
    ),
    (
        "modelo",
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            max_features=0.7,
            random_state=42,
            n_jobs=-1
        )
    )
])

# El clasificador aprende a distinguir producción cero de producción positiva.
clasificador_produccion.fit(X_train, y_train_positivo)

# En validación obtenemos tanto la clase estimada como
# la probabilidad de producción positiva.
clase_predicha = clasificador_produccion.predict(X_validation)
probabilidad_positiva = clasificador_produccion.predict_proba(X_validation)[:, 1]

# Resumimos la clasificación usando el umbral predeterminado de 0.5.
# Más adelante evaluaremos cómo combinar las probabilidades con la regresión.
matriz_confusion = confusion_matrix(
    y_validation_positivo,
    clase_predicha,
    labels=[0, 1]
)

display(
    pd.DataFrame(
        matriz_confusion,
        index=["Real: cero", "Real: positiva"],
        columns=["Predicho: cero", "Predicho: positiva"]
    )
)

print(f"Balanced accuracy: {balanced_accuracy_score(y_validation_positivo, clase_predicha):.3f}")
print(f"Precisión para producción positiva: {precision_score(y_validation_positivo, clase_predicha, zero_division=0):.3f}")
print(f"Recall para producción positiva: {recall_score(y_validation_positivo, clase_predicha, zero_division=0):.3f}")
print(f"F1 para producción positiva: {f1_score(y_validation_positivo, clase_predicha, zero_division=0):.3f}")
print(f"Probabilidad positiva promedio estimada: {probabilidad_positiva.mean():.3f}")

Distribución de las clases en entrenamiento:


,Registros
prod_pet_mes_siguiente,
Producción positiva,33601
Producción igual a cero,15851


,Predicho: cero,Predicho: positiva
Real: cero,4156,159
Real: positiva,157,9848


Balanced accuracy: 0.974
Precisión para producción positiva: 0.984
Recall para producción positiva: 0.984
F1 para producción positiva: 0.984
Probabilidad positiva promedio estimada: 0.692


#### 14.2. Segundo componente: regresión para los casos positivos

El clasificador estima si la producción de petróleo del mes siguiente será cero o positiva. El segundo componente debe estimar la cantidad producida cuando el objetivo es positivo.

Para entrenarlo utilizamos únicamente las filas de entrenamiento con `prod_pet_mes_siguiente` mayor que cero. De esta manera, el regresor aprende la cantidad de producción dentro de los casos positivos, sin intentar modelar también los ceros.

En validación combinamos las salidas de ambos componentes de dos maneras. En la primera, usamos la clase predicha para decidir entre cero y la cantidad estimada por el regresor. En la segunda, ponderamos la cantidad estimada por la probabilidad de producción positiva.

Comparamos ambas estrategias con la persistencia y los modelos de regresión ya evaluados, utilizando las mismas observaciones de validación. El conjunto de test no se utiliza en esta comparación.

In [16]:
from sklearn.base import clone

# Seleccionamos los ejemplos de entrenamiento con producción futura positiva.
mascara_positiva_train = y_train > 0

X_train_positivo = X_train.loc[mascara_positiva_train]
y_train_positivo_cantidad = y_train.loc[mascara_positiva_train]

if X_train_positivo.empty:
    raise ValueError("No hay ejemplos positivos para entrenar el regresor.")

print(f"Ejemplos positivos para entrenar el regresor: {len(X_train_positivo):,}")
print(
    "Producción mínima del grupo positivo: "
    f"{y_train_positivo_cantidad.min():.2f} m³"
)

# Reutilizamos el Random Forest candidato, pero lo ajustamos
# únicamente con los ejemplos de producción positiva.
regresor_positivo = clone(modelo_rf_candidato)
regresor_positivo.fit(X_train_positivo, y_train_positivo_cantidad)

# Estimamos la cantidad para todos los registros de validación.
# La combinación con el clasificador decidirá cómo usar estas cantidades.
y_pred_cantidad_positiva = regresor_positivo.predict(X_validation)

# Estrategia 1: usamos la clase predicha (umbral predeterminado 0.5).
y_pred_dos_etapas_clase = np.where(
    clase_predicha == 1,
    y_pred_cantidad_positiva,
    0
)

# Estrategia 2: ponderamos la cantidad por la probabilidad de producción positiva.
y_pred_dos_etapas_probabilidad = (
    probabilidad_positiva * y_pred_cantidad_positiva
)

# Comparamos en las mismas filas de validación.
predicciones_dos_etapas = {
    "Persistencia": y_pred_referencia,
    "Random Forest regresión": y_pred_modelo,
    "Dos etapas: decisión por clase": y_pred_dos_etapas_clase,
    "Dos etapas: ponderada por probabilidad": y_pred_dos_etapas_probabilidad,
}

resultados_dos_etapas = pd.DataFrame({
    "MAE (m³)": {
        nombre: mean_absolute_error(y_validation, prediccion)
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
    "RMSE (m³)": {
        nombre: np.sqrt(mean_squared_error(y_validation, prediccion))
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
    "R²": {
        nombre: r2_score(y_validation, prediccion)
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
    "Predicciones negativas": {
        nombre: (prediccion < 0).sum()
        for nombre, prediccion in predicciones_dos_etapas.items()
    },
})

display(resultados_dos_etapas.round(3))

Ejemplos positivos para entrenar el regresor: 33,601
Producción mínima del grupo positivo: 0.00 m³


,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,160.039,506.071,0.850,0
Random Forest regresión,128.364,368.283,0.920,0
Dos etapas: decisión por clase,126.950,367.603,0.921,0
Dos etapas: ponderada por probabilidad,127.985,365.949,0.921,0


### 15. Comparación temporal de la estrategia en dos etapas

En la validación de enero a marzo de 2026, las variantes de dos etapas obtuvieron métricas ligeramente mejores que el Random Forest de regresión directa. Como la diferencia fue pequeña, queremos comprobar si se mantiene en distintos períodos.

Para eso repetimos la comparación en los mismos pliegues temporales definidos dentro del conjunto de entrenamiento. En cada pliegue volvemos a entrenar desde cero el Random Forest directo, el clasificador y el regresor para casos positivos. El regresor de la estrategia en dos etapas utiliza únicamente observaciones de entrenamiento cuyo objetivo es positivo; el bloque posterior se reserva para evaluar las predicciones.

Comparamos la referencia de persistencia, el Random Forest directo y las dos formas de combinar el clasificador y el regresor. Utilizamos MAE como métrica principal y RMSE como complemento. No usamos el conjunto de test para elegir entre estrategias.

In [17]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline

resultados_cv_dos_etapas = []
predicciones_cv = {
    "Persistencia": {"real": [], "predicha": []},
    "Random Forest directo": {"real": [], "predicha": []},
    "Dos etapas: decisión por clase": {"real": [], "predicha": []},
    "Dos etapas: ponderada por probabilidad": {"real": [], "predicha": []},
}

for pliegue in pliegues_temporales:
    fin_entrenamiento = pd.Timestamp(pliegue["Entrenamiento hasta"])
    inicio_validacion = pd.Timestamp(pliegue["Validación desde"])
    fin_validacion = pd.Timestamp(pliegue["Validación hasta"])

    mascara_entrenamiento = train["mes_objetivo"] <= fin_entrenamiento
    mascara_validacion = train["mes_objetivo"].between(
        inicio_validacion,
        fin_validacion
    )

    datos_entrenamiento = train.loc[mascara_entrenamiento]
    datos_validacion = train.loc[mascara_validacion]

    if datos_entrenamiento.empty or datos_validacion.empty:
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no tiene filas suficientes."
        )

    if (
        datos_entrenamiento["mes_objetivo"].max()
        >= datos_validacion["mes_objetivo"].min()
    ):
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no respeta el orden temporal."
        )

    X_entrenamiento_pliegue = datos_entrenamiento[features]
    y_entrenamiento_pliegue = datos_entrenamiento[objetivo]
    X_validacion_pliegue = datos_validacion[features]
    y_real_pliegue = datos_validacion[objetivo].to_numpy()

    # Referencia: repetir la producción del mes de corte.
    y_persistencia_pliegue = datos_validacion["prod_pet"].to_numpy()

    # Modelo directo: predice la producción futura en un solo paso.
    modelo_directo_pliegue = clone(modelo_rf_candidato)
    modelo_directo_pliegue.fit(
        X_entrenamiento_pliegue,
        y_entrenamiento_pliegue
    )
    y_directo_pliegue = modelo_directo_pliegue.predict(
        X_validacion_pliegue
    )

    # Clasificador: aprende si el objetivo futuro es cero o positivo.
    y_clase_entrenamiento = (y_entrenamiento_pliegue > 0).astype(int)

    clasificador_pliegue = Pipeline([
        (
            "preprocesamiento",
            clone(modelo_regresion.named_steps["preprocesamiento"])
        ),
        (
            "modelo",
            RandomForestClassifier(
                n_estimators=200,
                min_samples_leaf=5,
                max_features=0.7,
                random_state=42,
                n_jobs=-1
            )
        )
    ])
    clasificador_pliegue.fit(
        X_entrenamiento_pliegue,
        y_clase_entrenamiento
    )

    # Regresor: aprende la cantidad usando solo objetivos positivos.
    mascara_positiva = y_entrenamiento_pliegue > 0
    X_positivos_pliegue = X_entrenamiento_pliegue.loc[mascara_positiva]
    y_positivos_pliegue = y_entrenamiento_pliegue.loc[mascara_positiva]

    if X_positivos_pliegue.empty:
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no tiene objetivos positivos "
            "para entrenar el regresor."
        )

    if not (y_positivos_pliegue > 0).all():
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} contiene objetivos no positivos "
            "en el entrenamiento del regresor."
        )

    regresor_positivos_pliegue = clone(modelo_rf_candidato)
    regresor_positivos_pliegue.fit(
        X_positivos_pliegue,
        y_positivos_pliegue
    )

    # Calculamos la clase y la probabilidad de producción positiva.
    clase_predicha_pliegue = clasificador_pliegue.predict(
        X_validacion_pliegue
    )
    probabilidades = clasificador_pliegue.predict_proba(
        X_validacion_pliegue
    )
    indice_clase_positiva = list(
        clasificador_pliegue.named_steps["modelo"].classes_
    ).index(1)
    probabilidad_positiva_pliegue = probabilidades[:, indice_clase_positiva]

    # El regresor estima la cantidad condicional para cada fila.
    cantidad_positiva_pliegue = regresor_positivos_pliegue.predict(
        X_validacion_pliegue
    )

    # Combinación 1: clase cero -> cero; clase positiva -> cantidad estimada.
    y_dos_etapas_clase = np.where(
        clase_predicha_pliegue == 1,
        cantidad_positiva_pliegue,
        0
    )

    # Combinación 2: cantidad estimada ponderada por la probabilidad positiva.
    y_dos_etapas_probabilidad = (
        probabilidad_positiva_pliegue * cantidad_positiva_pliegue
    )

    predicciones_pliegue = {
        "Persistencia": y_persistencia_pliegue,
        "Random Forest directo": y_directo_pliegue,
        "Dos etapas: decisión por clase": y_dos_etapas_clase,
        "Dos etapas: ponderada por probabilidad": y_dos_etapas_probabilidad,
    }

    # Guardamos métricas por pliegue y predicciones para el resumen global.
    for nombre, y_predicha in predicciones_pliegue.items():
        resultados_cv_dos_etapas.append({
            "Pliegue": pliegue["Pliegue"],
            "Método": nombre,
            "Filas de validación": len(y_real_pliegue),
            "MAE (m³)": mean_absolute_error(y_real_pliegue, y_predicha),
            "RMSE (m³)": np.sqrt(
                mean_squared_error(y_real_pliegue, y_predicha)
            ),
        })

        predicciones_cv[nombre]["real"].extend(y_real_pliegue)
        predicciones_cv[nombre]["predicha"].extend(y_predicha)

tabla_cv_por_pliegue = (
    pd.DataFrame(resultados_cv_dos_etapas)
    .set_index(["Pliegue", "Método"])
    .sort_index()
)

print("Resultados por pliegue:")
display(tabla_cv_por_pliegue.round(2))

# Calculamos métricas agrupadas sobre todas las predicciones de los pliegues.
resumen_cv = []

for nombre, datos in predicciones_cv.items():
    y_real_total = np.asarray(datos["real"])
    y_predicha_total = np.asarray(datos["predicha"])

    resumen_cv.append({
        "Método": nombre,
        "Registros": len(y_real_total),
        "MAE agrupado (m³)": mean_absolute_error(
            y_real_total,
            y_predicha_total
        ),
        "RMSE agrupado (m³)": np.sqrt(
            mean_squared_error(y_real_total, y_predicha_total)
        ),
    })

print("\nMétricas agrupadas de los pliegues:")
display(
    pd.DataFrame(resumen_cv)
    .set_index("Método")
    .sort_values("MAE agrupado (m³)")
    .round(2)
)

Resultados por pliegue:


Filas de validación  MAE (m³)  \
Pliegue Método                                                                  
1       Dos etapas: decisión por clase                         9118    130.74   
        Dos etapas: ponderada por probabilidad                 9118    132.61   
        Persistencia                                           9118    161.31   
        Random Forest directo                                  9118    130.68   
2       Dos etapas: decisión por clase                         9307    127.11   
        Dos etapas: ponderada por probabilidad                 9307    129.06   
        Persistencia                                           9307    166.55   
        Random Forest directo                                  9307    128.30   
3       Dos etapas: decisión por clase                         4704    134.00   
        Dos etapas: ponderada por probabilidad                 4704    135.20   
        Persistencia                                           4704    163.76   
        Random Forest directo                                  4704    135.19   

                                                RMSE (m³)  
Pliegue Método                                             
1       Dos etapas: decisión por clase             394.65  
        Dos etapas: ponderada por probabilidad     394.99  
        Persistencia                               522.44  
        Random Forest directo                      391.88  
2       Dos etapas: decisión por clase             387.70  
        Dos etapas: ponderada por probabilidad     387.63  
        Persistencia                               559.03  
        Random Forest directo                      388.84  
3       Dos etapas: decisión por clase             407.84  
        Dos etapas: ponderada por probabilidad     410.13  
        Persistencia                               562.44  
        Random Forest directo                      407.07


Métricas agrupadas de los pliegues:


,Registros,MAE agrupado (m³),RMSE agrupado (m³)
Método,,,
Dos etapas: decisión por clase,23129,129.94,394.60
Random Forest directo,23129,130.64,393.80
Dos etapas: ponderada por probabilidad,23129,131.71,395.20
Persistencia,23129,163.92,545.62


#### Interpretación de la comparación temporal

En los tres pliegues temporales, los modelos de Random Forest obtuvieron menores valores de MAE y RMSE que la referencia de persistencia. Esto indica que ambas estrategias de regresión encontraron patrones útiles en los datos para anticipar la producción del mes siguiente.

Al comparar la regresión directa con la estrategia de dos etapas por clase, esta última obtuvo el menor MAE agrupado (129,94 m³ frente a 130,64 m³). La diferencia fue pequeña, de aproximadamente 0,5 %. En cambio, el Random Forest directo obtuvo un RMSE ligeramente menor (393,80 m³ frente a 394,60 m³). La estrategia ponderada por probabilidad no mejoró las métricas del modelo directo en estos pliegues.

Por lo tanto, no observamos una ventaja concluyente de la estrategia de dos etapas frente a la regresión directa. Mantenemos el Random Forest directo como referencia principal por su menor complejidad y su menor RMSE, y conservamos la variante por clase como alternativa para considerar si priorizamos el MAE o la identificación explícita de meses sin producción. Estas diferencias deben interpretarse como pequeñas y no como una mejora definitiva.

### 16. Comparación de ventanas de entrenamiento

En las secciones anteriores trabajamos con la partición temporal definida en la notebook 03. Esa configuración utiliza los objetivos de producción desde febrero hasta diciembre de 2025 para entrenar, los objetivos de enero a marzo de 2026 para validar y los objetivos de abril a julio de 2026 para realizar la evaluación final.

Aunque la tabla original se mantiene sin cambios, en esta sección analizamos si conviene ampliar la ventana de entrenamiento incorporando objetivos de años anteriores a 2025.

La motivación es evaluar si una mayor cantidad de ejemplos históricos ayuda al modelo a aprender patrones más generales. Sin embargo, no suponemos que agregar datos antiguos necesariamente mejore el resultado, porque la cantidad de pozos, el nivel de producción y la composición de los registros cambian a lo largo del tiempo.

Para realizar una comparación justa:

- mantenemos las mismas variables predictoras;
- utilizamos el mismo Random Forest candidato;
- conservamos los mismos períodos de validación;
- no usamos el conjunto de test para elegir la ventana;
- modificamos únicamente el período utilizado para entrenar.

#### 16.1. Datos históricos disponibles

En la notebook 03 exportamos un archivo adicional con todos los registros cuyo objetivo está disponible hasta diciembre de 2025. Este archivo incluye información desde febrero de 2006 y se utiliza únicamente para comparar ventanas de entrenamiento.

La partición original continúa disponible y no se reemplaza. De esta forma podemos comparar la configuración actual con alternativas que incorporan más años de historia.

In [18]:
# Cargamos los ejemplos históricos candidatos a entrenamiento.
ruta_entrenamiento_historico = (
    RUTA_PROCESSED / "entrenamiento_historico_candidato.csv"
)

if not ruta_entrenamiento_historico.is_file():
    raise FileNotFoundError(
        f"No se encontró el archivo: {ruta_entrenamiento_historico}. "
        "Ejecutá primero la celda de exportación de la notebook 03."
    )

entrenamiento_historico = pd.read_csv(
    ruta_entrenamiento_historico,
    parse_dates=["fecha", "mes_objetivo"],
    low_memory=False,
)

# Comprobamos que estén presentes los predictores y el objetivo.
columnas_necesarias = set(features + [objetivo])
columnas_faltantes = sorted(
    columnas_necesarias - set(entrenamiento_historico.columns)
)

if columnas_faltantes:
    raise KeyError(
        "Faltan columnas necesarias en el archivo histórico: "
        f"{columnas_faltantes}"
    )

if entrenamiento_historico[objetivo].isna().any():
    raise ValueError(
        "El archivo histórico contiene valores faltantes en el objetivo."
    )

if entrenamiento_historico["mes_objetivo"].max() > pd.Timestamp("2025-12-01"):
    raise ValueError(
        "El archivo incluye objetivos posteriores a diciembre de 2025."
    )

print(f"Filas históricas disponibles: {len(entrenamiento_historico):,}")
print(f"Pozos: {entrenamiento_historico['idpozo'].nunique():,}")
print(
    "Período de objetivos:",
    entrenamiento_historico["mes_objetivo"].min().strftime("%Y-%m"),
    "a",
    entrenamiento_historico["mes_objetivo"].max().strftime("%Y-%m"),
)
print(f"Variables predictoras presentes: {len(features)}")

Filas históricas disponibles: 381,069
Pozos: 4,725
Período de objetivos: 2006-02 a 2025-12
Variables predictoras presentes: 41


#### 16.2. Prueba técnica con un pliegue temporal

Antes de ejecutar la comparación completa, realizamos una prueba técnica utilizando solamente el primer pliegue temporal.

Comparamos:

- la ventana actual, que comienza en febrero de 2025;
- la ventana ampliada, que comienza en enero de 2020.

En esta prueba utilizamos 30 árboles en lugar de la configuración completa del Random Forest. El objetivo principal es estimar el tiempo de ejecución y verificar que el proceso funcione correctamente con un volumen de datos mayor.

Por lo tanto, estos resultados no se utilizan para seleccionar definitivamente el modelo. Funcionan como un diagnóstico inicial para decidir si resulta razonable ejecutar las comparaciones posteriores.

In [19]:
import time

# Prueba inicial: comparamos la ventana actual con la que comienza en 2020
# usando un solo pliegue temporal.
pliegue_prueba = pliegues_temporales[0]

fin_entrenamiento = pd.Timestamp(pliegue_prueba["Entrenamiento hasta"])
inicio_validacion = pd.Timestamp(pliegue_prueba["Validación desde"])
fin_validacion = pd.Timestamp(pliegue_prueba["Validación hasta"])

datos_validacion_prueba = train.loc[
    train["mes_objetivo"].between(inicio_validacion, fin_validacion)
].copy()

ventanas_prueba = {
    "Actual: desde 2025": pd.Timestamp("2025-02-01"),
    "Desde 2020": pd.Timestamp("2020-01-01"),
}

resultados_prueba = []

for nombre, inicio_ventana in ventanas_prueba.items():
    datos_entrenamiento_prueba = entrenamiento_historico.loc[
        entrenamiento_historico["mes_objetivo"].between(
            inicio_ventana,
            fin_entrenamiento,
        )
    ]

    print(
        f"Iniciando {nombre}: "
        f"{len(datos_entrenamiento_prueba):,} filas de entrenamiento...",
        flush=True,
    )

    modelo_prueba = clone(modelo_rf_candidato)

    # Usamos menos árboles solo para medir el tiempo de ejecución.
    # Estos resultados son una prueba técnica, no la comparación final.
    modelo_prueba.set_params(modelo__n_estimators=30)

    inicio = time.time()
    modelo_prueba.fit(
        datos_entrenamiento_prueba[features],
        datos_entrenamiento_prueba[objetivo],
    )
    segundos = time.time() - inicio

    prediccion = modelo_prueba.predict(
        datos_validacion_prueba[features]
    )

    resultados_prueba.append({
        "Ventana": nombre,
        "Filas de entrenamiento": len(datos_entrenamiento_prueba),
        "Filas de validación": len(datos_validacion_prueba),
        "MAE de prueba (m³)": mean_absolute_error(
            datos_validacion_prueba[objetivo],
            prediccion,
        ),
        "Tiempo de ajuste (segundos)": round(segundos, 1),
    })

    print(f"Terminado {nombre} en {segundos:.1f} segundos.", flush=True)

display(pd.DataFrame(resultados_prueba).set_index("Ventana").round(2))

Iniciando Actual: desde 2025: 26,323 filas de entrenamiento...


Terminado Actual: desde 2025 en 2.1 segundos.


Iniciando Desde 2020: 225,496 filas de entrenamiento...


Terminado Desde 2020 en 43.2 segundos.


,Filas de entrenamiento,Filas de validación,MAE de prueba (m³),Tiempo de ajuste (segundos)
Ventana,,,,
Actual: desde 2025,26323,9118,133.37,2.1
Desde 2020,225496,9118,127.84,43.2


La ventana desde 2020 utilizó una cantidad considerablemente mayor de registros y, en esta primera prueba, obtuvo un MAE menor que la ventana actual.

Como se utilizaron solamente 30 árboles y un único pliegue, el resultado se considera preliminar. La diferencia observada justifica continuar con una comparación más completa en los tres pliegues temporales.

#### 16.3. Comparación preliminar en los tres pliegues temporales

Luego de verificar que el proceso funciona correctamente, ampliamos la comparación a los tres pliegues temporales utilizados anteriormente.

En cada pliegue:

- las filas de validación son iguales para ambas ventanas;
- los objetivos de validación siempre son posteriores a los objetivos de entrenamiento;
- se utiliza el mismo Random Forest;
- solamente cambia la fecha inicial del entrenamiento.

En esta etapa seguimos utilizando 30 árboles para reducir el tiempo de ejecución. Por eso, los resultados se consideran preliminares y se utilizarán para observar si la mejora se mantiene a lo largo de distintos períodos.

In [20]:
# Comparamos la ventana actual con la ampliada desde 2020.
# Usamos 30 árboles para mantener acotado el tiempo de ejecución.
ventanas_prueba_temporal = {
    "Actual: desde 2025": pd.Timestamp("2025-02-01"),
    "Desde 2020": pd.Timestamp("2020-01-01"),
}

resultados_temporales_ventanas = []
predicciones_temporales_ventanas = {
    nombre: {"real": [], "prediccion": []}
    for nombre in ventanas_prueba_temporal
}

for pliegue in pliegues_temporales:
    fin_entrenamiento = pd.Timestamp(pliegue["Entrenamiento hasta"])
    inicio_validacion = pd.Timestamp(pliegue["Validación desde"])
    fin_validacion = pd.Timestamp(pliegue["Validación hasta"])

    # La validación es idéntica para las dos ventanas.
    datos_validacion_prueba = train.loc[
        train["mes_objetivo"].between(inicio_validacion, fin_validacion)
    ].copy()

    if datos_validacion_prueba.empty:
        raise ValueError(
            f"El pliegue {pliegue['Pliegue']} no tiene filas de validación."
        )

    y_real = datos_validacion_prueba[objetivo].to_numpy()

    for nombre, inicio_ventana in ventanas_prueba_temporal.items():
        datos_entrenamiento_prueba = entrenamiento_historico.loc[
            entrenamiento_historico["mes_objetivo"].between(
                inicio_ventana,
                fin_entrenamiento,
            )
        ]

        if datos_entrenamiento_prueba.empty:
            raise ValueError(
                f"No hay filas de entrenamiento para '{nombre}' "
                f"en el pliegue {pliegue['Pliegue']}."
            )

        print(
            f"Pliegue {pliegue['Pliegue']} | {nombre}: "
            f"{len(datos_entrenamiento_prueba):,} filas de entrenamiento",
            flush=True,
        )

        modelo_prueba_temporal = clone(modelo_rf_candidato)
        modelo_prueba_temporal.set_params(modelo__n_estimators=30)

        inicio_ajuste = time.time()
        modelo_prueba_temporal.fit(
            datos_entrenamiento_prueba[features],
            datos_entrenamiento_prueba[objetivo],
        )
        segundos_ajuste = time.time() - inicio_ajuste

        y_predicho = modelo_prueba_temporal.predict(
            datos_validacion_prueba[features]
        )

        resultados_temporales_ventanas.append({
            "Ventana": nombre,
            "Pliegue": pliegue["Pliegue"],
            "Filas de entrenamiento": len(datos_entrenamiento_prueba),
            "Filas de validación": len(datos_validacion_prueba),
            "MAE (m³)": mean_absolute_error(y_real, y_predicho),
            "RMSE (m³)": np.sqrt(mean_squared_error(y_real, y_predicho)),
            "Tiempo de ajuste (segundos)": round(segundos_ajuste, 1),
        })

        predicciones_temporales_ventanas[nombre]["real"].extend(y_real)
        predicciones_temporales_ventanas[nombre]["prediccion"].extend(y_predicho)

resultados_detallados_ventanas = (
    pd.DataFrame(resultados_temporales_ventanas)
    .set_index(["Ventana", "Pliegue"])
    .round(2)
)

display(resultados_detallados_ventanas)

# Agrupamos las predicciones de los tres períodos para obtener
# una métrica conjunta por ventana de entrenamiento.
resumen_temporal_ventanas = []

for nombre, datos in predicciones_temporales_ventanas.items():
    y_real = np.asarray(datos["real"])
    y_predicho = np.asarray(datos["prediccion"])

    resumen_temporal_ventanas.append({
        "Ventana": nombre,
        "Filas de validación agrupadas": len(y_real),
        "MAE agrupado (m³)": mean_absolute_error(y_real, y_predicho),
        "RMSE agrupado (m³)": np.sqrt(
            mean_squared_error(y_real, y_predicho)
        ),
    })

display(
    pd.DataFrame(resumen_temporal_ventanas)
    .set_index("Ventana")
    .round(2)
)

Pliegue 1 | Actual: desde 2025: 26,323 filas de entrenamiento


Pliegue 1 | Desde 2020: 225,496 filas de entrenamiento


Pliegue 2 | Actual: desde 2025: 35,441 filas de entrenamiento


Pliegue 2 | Desde 2020: 234,614 filas de entrenamiento


Pliegue 3 | Actual: desde 2025: 44,748 filas de entrenamiento


Pliegue 3 | Desde 2020: 243,921 filas de entrenamiento


,,Filas de entrenamiento,Filas de validación,MAE (m³),RMSE (m³),Tiempo de ajuste (segundos)
Ventana,Pliegue,,,,,
Actual: desde 2025,1,26323,9118,133.37,396.65,2.4
Desde 2020,1,225496,9118,127.84,379.09,39.9
Actual: desde 2025,2,35441,9307,129.31,391.07,3.0
Desde 2020,2,234614,9307,130.06,395.37,42.4
Actual: desde 2025,3,44748,4704,136.06,407.74,4.3
Desde 2020,3,243921,4704,133.49,402.14,45.0


,Filas de validación agrupadas,MAE agrupado (m³),RMSE agrupado (m³)
Ventana,,,
Actual: desde 2025,23129,132.29,396.71
Desde 2020,23129,129.88,390.44


La comparación preliminar mostró que la ventana desde 2020 obtuvo mejores resultados agrupados que la ventana actual.

La mejora también se observó en los tres pliegues, aunque fue más importante en el primer período y más pequeña en los siguientes. Esto sugiere que la información histórica adicional puede ser útil, pero que su impacto no es uniforme en todos los meses.

Como esta prueba se realizó con 30 árboles, todavía no utilizamos estos valores para seleccionar el modelo final. La siguiente etapa consiste en repetir la comparación con la configuración completa del Random Forest y sobre el conjunto de validación principal.

#### 16.4. Confirmación con la configuración completa

Después de observar una mejora preliminar al incorporar objetivos desde 2020, repetimos la comparación utilizando la configuración completa del Random Forest candidato.

En esta etapa comparamos las dos ventanas sobre el mismo conjunto de validación, correspondiente a los objetivos de enero a marzo de 2026:

- ventana actual: objetivos desde febrero de 2025;
- ventana ampliada: objetivos desde enero de 2020.

La persistencia se incluye como referencia. El conjunto de test no se utiliza para seleccionar la ventana, sino que se mantiene reservado para informar posteriormente el desempeño final del modelo elegido.

In [21]:
# Comparamos las dos ventanas en la misma validación de enero a marzo de 2026.
ventanas_confirmacion = {
    "Actual: desde 2025": pd.Timestamp("2025-02-01"),
    "Desde 2020": pd.Timestamp("2020-01-01"),
}

resultados_confirmacion = []

y_real_validacion = validation[objetivo].to_numpy()
y_persistencia_validacion = validation["prod_pet"].to_numpy()

for nombre, inicio_ventana in ventanas_confirmacion.items():
    datos_entrenamiento = entrenamiento_historico.loc[
        entrenamiento_historico["mes_objetivo"].between(
            inicio_ventana,
            pd.Timestamp("2025-12-01"),
        )
    ]

    print(
        f"Iniciando {nombre}: {len(datos_entrenamiento):,} filas...",
        flush=True,
    )

    modelo_confirmacion = clone(modelo_rf_candidato)

    inicio_ajuste = time.time()
    modelo_confirmacion.fit(
        datos_entrenamiento[features],
        datos_entrenamiento[objetivo],
    )
    segundos_ajuste = time.time() - inicio_ajuste

    y_predicho = modelo_confirmacion.predict(validation[features])

    resultados_confirmacion.append({
        "Ventana": nombre,
        "Filas de entrenamiento": len(datos_entrenamiento),
        "Filas de validación": len(validation),
        "MAE (m³)": mean_absolute_error(y_real_validacion, y_predicho),
        "RMSE (m³)": np.sqrt(
            mean_squared_error(y_real_validacion, y_predicho)
        ),
        "R²": r2_score(y_real_validacion, y_predicho),
        "Tiempo de ajuste (segundos)": round(segundos_ajuste, 1),
    })

    print(
        f"Terminado {nombre} en {segundos_ajuste:.1f} segundos.",
        flush=True,
    )

# Agregamos la referencia de persistencia para facilitar la comparación.
resultados_confirmacion.append({
    "Ventana": "Persistencia",
    "Filas de entrenamiento": 0,
    "Filas de validación": len(validation),
    "MAE (m³)": mean_absolute_error(
        y_real_validacion,
        y_persistencia_validacion,
    ),
    "RMSE (m³)": np.sqrt(
        mean_squared_error(y_real_validacion, y_persistencia_validacion)
    ),
    "R²": r2_score(
        y_real_validacion,
        y_persistencia_validacion,
    ),
    "Tiempo de ajuste (segundos)": 0,
})

display(
    pd.DataFrame(resultados_confirmacion)
    .set_index("Ventana")
    .round(3)
)

Iniciando Actual: desde 2025: 49,452 filas...


Terminado Actual: desde 2025 en 28.8 segundos.


Iniciando Desde 2020: 248,625 filas...


Terminado Desde 2020 en 279.1 segundos.


,Filas de entrenamiento,Filas de validación,MAE (m³),RMSE (m³),R²,Tiempo de ajuste (segundos)
Ventana,,,,,,
Actual: desde 2025,49452,14320,127.156,364.622,0.922,28.8
Desde 2020,248625,14320,125.838,360.401,0.924,279.1
Persistencia,0,14320,160.039,506.071,0.850,0.0


La ventana desde 2020 obtuvo un desempeño ligeramente mejor que la ventana actual en el conjunto de validación.

El MAE disminuyó de 127,156 m³ a 125,838 m³, el RMSE disminuyó de 364,622 m³ a 360,401 m³ y el R² aumentó de 0,922 a 0,924.

La mejora es moderada, pero coincide con la tendencia observada en los pliegues temporales. Además, ambas ventanas superan ampliamente a la referencia de persistencia, que obtuvo un MAE de 160,039 m³ y un RMSE de 506,071 m³.

El tiempo de ajuste se informa como dato operativo. La ventana desde 2020 requiere más tiempo porque utiliza una cantidad mayor de registros, pero esto no afecta la calidad de las predicciones una vez que el modelo fue entrenado.

#### 16.5. Selección de la ventana de entrenamiento

A partir de las comparaciones realizadas, seleccionamos como ventana final de entrenamiento el período comprendido entre enero de 2020 y diciembre de 2025.

La ventana desde 2020 obtuvo el mejor desempeño en la validación principal:

- MAE: 125,838 m³;
- RMSE: 360,401 m³;
- R²: 0,924.

La ventana actual, que utiliza objetivos desde 2025, obtuvo:

- MAE: 127,156 m³;
- RMSE: 364,622 m³;
- R²: 0,922.

La diferencia es moderada, pero la ventana desde 2020 también presentó mejores resultados en la comparación con los pliegues temporales. Por este motivo, consideramos que incorporar información supervisada de los años 2020 a 2024 aporta una mejora pequeña pero consistente.

No seleccionamos la ventana desde 2025 como configuración final porque utiliza una cantidad menor de ejemplos y presentó métricas ligeramente peores. Sin embargo, la conservamos como referencia para medir el efecto de ampliar el entrenamiento.

Tampoco elegimos directamente una ventana desde 2006. Aunque contiene más registros, los primeros años presentan una distribución diferente de la producción, con una proporción mucho mayor de valores iguales a cero y niveles medios de producción más bajos. Además, esos períodos pueden representar condiciones operativas y una composición de pozos distintas de las de los años más recientes. Por este motivo, incluir toda la historia no garantiza una mejor representación del contexto actual.

La comparación desde 2020 ofrece un equilibrio entre cantidad de datos y actualidad de la información. Permite ampliar considerablemente el entrenamiento sin incorporar los períodos más antiguos, que muestran diferencias importantes en su distribución.

La selección se realizó utilizando únicamente los resultados de validación y de los pliegues temporales. El conjunto de test no se utiliza para elegir la ventana y se mantiene reservado para la evaluación final.

En conclusión, el modelo final de regresión se entrenará con los objetivos comprendidos entre enero de 2020 y diciembre de 2025. La persistencia continúa utilizándose como baseline para evaluar si el modelo supera una referencia sencilla.

### 17. Entrenamiento final del modelo de regresión

Una vez seleccionada la ventana de entrenamiento, ajustamos el modelo final de regresión utilizando todos los registros disponibles desde enero de 2020 hasta diciembre de 2025.

En esta etapa no se modifican las variables predictoras ni los hiperparámetros. El objetivo es entrenar la configuración seleccionada y medir una sola vez su desempeño sobre el conjunto de test, que representa meses posteriores a los utilizados para ajustar el modelo.

El conjunto de test se mantiene separado del entrenamiento y no se utiliza para tomar nuevas decisiones sobre la ventana o los hiperparámetros.

### 17.1. Preparación del conjunto final de entrenamiento

Seleccionamos los registros históricos cuyos objetivos están comprendidos entre enero de 2020 y diciembre de 2025. Estos registros se utilizarán para ajustar el Random Forest final.

Validamos también que:

- el conjunto no esté vacío;
- el objetivo no tenga valores faltantes;
- el período final de entrenamiento no se superponga con validación ni con test;
- las variables predictoras requeridas estén presentes.

In [22]:
# Seleccionamos los registros para el entrenamiento final.
# El modelo utilizará los objetivos desde enero de 2020
# hasta diciembre de 2025.
entrenamiento_final_regresion = entrenamiento_historico.loc[
    entrenamiento_historico["mes_objetivo"].between(
        "2020-01-01",
        "2025-12-01",
    )
].copy()

if entrenamiento_final_regresion.empty:
    raise ValueError(
        "No se encontraron registros para el entrenamiento final."
    )

if entrenamiento_final_regresion[objetivo].isna().any():
    raise ValueError(
        "El entrenamiento final contiene valores faltantes en el objetivo."
    )

columnas_faltantes_final = sorted(
    set(features) - set(entrenamiento_final_regresion.columns)
)

if columnas_faltantes_final:
    raise KeyError(
        "Faltan variables predictoras en el entrenamiento final: "
        f"{columnas_faltantes_final}"
    )

if (
    entrenamiento_final_regresion["mes_objetivo"].max()
    >= validation["mes_objetivo"].min()
):
    raise ValueError(
        "El entrenamiento final se superpone con el período de validación."
    )

print(
    "Período de objetivos:",
    entrenamiento_final_regresion["mes_objetivo"].min().strftime("%Y-%m"),
    "a",
    entrenamiento_final_regresion["mes_objetivo"].max().strftime("%Y-%m"),
)
print(
    f"Filas de entrenamiento final: "
    f"{len(entrenamiento_final_regresion):,}"
)
print(
    f"Pozos de entrenamiento final: "
    f"{entrenamiento_final_regresion['idpozo'].nunique():,}"
)
print(f"Variables predictoras: {len(features)}")

Período de objetivos: 2020-01 a 2025-12
Filas de entrenamiento final: 248,625
Pozos de entrenamiento final: 4,718
Variables predictoras: 41


#### 17.2. Ajuste del Random Forest final

Ajustamos una copia del Random Forest candidato con el conjunto final de entrenamiento. Se mantienen los hiperparámetros elegidos por `GridSearchCV` en la sección 10.

El modelo se entrena únicamente con los registros seleccionados desde 2020 hasta 2025. La validación y el test no participan de este ajuste.

In [23]:
# Creamos una copia independiente del modelo candidato.
# De esta manera no modificamos los modelos utilizados
# durante las comparaciones anteriores.
modelo_rf_final = clone(modelo_rf_candidato)

print(
    "Iniciando entrenamiento del modelo final de regresión...",
    flush=True,
)

inicio_entrenamiento_final = time.time()

modelo_rf_final.fit(
    entrenamiento_final_regresion[features],
    entrenamiento_final_regresion[objetivo],
)

segundos_entrenamiento_final = (
    time.time() - inicio_entrenamiento_final
)

print(
    "Entrenamiento finalizado en "
    f"{segundos_entrenamiento_final:.1f} segundos.",
    flush=True,
)

print("Modelo: Random Forest regresor")
print("Ventana de objetivos: 2020-01 a 2025-12")
print(
    f"Filas utilizadas: "
    f"{len(entrenamiento_final_regresion):,}"
)
print(f"Variables predictoras: {len(features)}")

Iniciando entrenamiento del modelo final de regresión...


Entrenamiento finalizado en 277.3 segundos.


Modelo: Random Forest regresor
Ventana de objetivos: 2020-01 a 2025-12
Filas utilizadas: 248,625
Variables predictoras: 41


### 18. Evaluación final en el conjunto de test

Una vez entrenado el modelo final, evaluamos su comportamiento sobre el conjunto de test. Este conjunto contiene objetivos posteriores a los utilizados para el entrenamiento y no participa en el ajuste del modelo.

La evaluación se realiza comparando el Random Forest final contra la referencia de persistencia. La persistencia estima que la producción del mes siguiente será igual a la producción del mes de corte.

En esta etapa calculamos MAE, RMSE y R². También verificamos si el modelo genera predicciones negativas, ya que la producción no puede tomar valores menores que cero.

El conjunto de test se utiliza una sola vez para informar el desempeño final del modelo seleccionado.

In [24]:
# Preparamos el objetivo real del conjunto de test.
y_test_final = test[objetivo].to_numpy()

if np.isnan(y_test_final).any():
    raise ValueError(
        "El objetivo del conjunto de test contiene valores faltantes."
    )

# Referencia de persistencia:
# la producción predicha es la del mes de corte.
y_pred_persistencia_test_final = test["prod_pet"].to_numpy()

if np.isnan(y_pred_persistencia_test_final).any():
    raise ValueError(
        "La variable prod_pet contiene valores faltantes en test."
    )

# Predicciones del Random Forest final.
y_pred_rf_final_test = modelo_rf_final.predict(
    test[features]
)

if not np.isfinite(y_pred_rf_final_test).all():
    raise ValueError(
        "El modelo final generó predicciones no finitas."
    )

resultados_test_final = pd.DataFrame(
    {
        "MAE (m³)": [
            mean_absolute_error(
                y_test_final,
                y_pred_persistencia_test_final,
            ),
            mean_absolute_error(
                y_test_final,
                y_pred_rf_final_test,
            ),
        ],
        "RMSE (m³)": [
            np.sqrt(
                mean_squared_error(
                    y_test_final,
                    y_pred_persistencia_test_final,
                )
            ),
            np.sqrt(
                mean_squared_error(
                    y_test_final,
                    y_pred_rf_final_test,
                )
            ),
        ],
        "R²": [
            r2_score(
                y_test_final,
                y_pred_persistencia_test_final,
            ),
            r2_score(
                y_test_final,
                y_pred_rf_final_test,
            ),
        ],
        "Predicciones negativas": [
            (y_pred_persistencia_test_final < 0).sum(),
            (y_pred_rf_final_test < 0).sum(),
        ],
    },
    index=["Persistencia", "Random Forest final"],
)

display(resultados_test_final.round(3))

,MAE (m³),RMSE (m³),R²,Predicciones negativas
Persistencia,166.195,551.710,0.820,0
Random Forest final,133.616,394.712,0.908,0


#### 18.1. Interpretación de la evaluación final

El Random Forest final obtuvo un MAE de 133,616 m³, un RMSE de 394,712 m³ y un R² de 0,908 en el conjunto de test. La referencia de persistencia obtuvo un MAE de 166,195 m³, un RMSE de 551,710 m³ y un R² de 0,820.

Por lo tanto, el modelo final redujo los errores respecto de la referencia y explicó una mayor proporción de la variabilidad de la producción futura. La mejora fue aproximadamente del 20 % en MAE y del 29 % en RMSE.

El resultado de test es algo inferior al de validación, donde el modelo había obtenido un MAE de 125,838 m³ y un RMSE de 360,401 m³. Esta diferencia es esperable porque el conjunto de test corresponde a meses posteriores y puede presentar condiciones distintas. De todos modos, el modelo mantuvo una ventaja clara frente a persistencia.

Además, no se observaron predicciones negativas. Esto es consistente con la naturaleza de la variable objetivo, que representa una cantidad de producción y no puede ser menor que cero.

Con estos resultados consideramos finalizada la selección y evaluación general del modelo de regresión. A partir de este punto no se modifican nuevamente sus hiperparámetros ni el inicio de su ventana de entrenamiento. Solo para generar la inferencia de agosto de 2026 reentrenamos la misma configuración incorporando los meses de validación y test (sección 20.1).

### 19. Análisis de errores por producción real

Las métricas generales resumen el desempeño del modelo, pero no muestran si el error se concentra en determinados tipos de casos. Por eso analizamos por separado los registros cuya producción real del mes siguiente fue igual a cero y aquellos cuya producción fue positiva.

Este análisis permite observar si el modelo tiende a sobreestimar los casos sin producción o si presenta mayores dificultades en los pozos que efectivamente producen.

In [25]:
# Construimos una tabla con la producción real y las predicciones
# del modelo final en el conjunto de test.
analisis_errores_test = pd.DataFrame({
    "Producción real": y_test_final,
    "Predicción Random Forest": y_pred_rf_final_test,
})

analisis_errores_test["Grupo"] = np.where(
    analisis_errores_test["Producción real"] == 0,
    "Producción real igual a cero",
    "Producción real positiva",
)

resultados_test_por_grupo_final = []

for grupo, datos in analisis_errores_test.groupby(
    "Grupo",
    sort=False,
):
    errores = (
        datos["Predicción Random Forest"]
        - datos["Producción real"]
    )

    resultados_test_por_grupo_final.append({
        "Grupo": grupo,
        "Registros": len(datos),
        "Porcentaje de test (%)": (
            len(datos)
            / len(analisis_errores_test)
            * 100
        ),
        "MAE (m³)": np.abs(errores).mean(),
        "RMSE (m³)": np.sqrt((errores ** 2).mean()),
        "Error promedio (m³)": errores.mean(),
        "Producción real media (m³)": (
            datos["Producción real"].mean()
        ),
        "Predicción media (m³)": (
            datos["Predicción Random Forest"].mean()
        ),
    })

resultados_test_por_grupo_final = (
    pd.DataFrame(resultados_test_por_grupo_final)
    .set_index("Grupo")
    .round(2)
)

display(resultados_test_por_grupo_final)

,Registros,Porcentaje de test (%),MAE (m³),RMSE (m³),Error promedio (m³),Producción real media (m³),Predicción media (m³)
Grupo,,,,,,,
Producción real positiva,13996,70.91,178.32,458.37,-7.08,868.28,861.21
Producción real igual a cero,5741,29.09,24.63,152.98,24.63,0.00,24.63


### 20. Inferencia para agosto de 2026

Una vez seleccionado, entrenado y evaluado el modelo final, lo aplicamos al conjunto de inferencia generado en la notebook 03.

Este conjunto contiene la información disponible hasta julio de 2026 para cada pozo. Por lo tanto, las predicciones generadas corresponden a la producción esperada durante agosto de 2026.

A diferencia del conjunto de test, en este caso todavía no conocemos la producción real de agosto. Por ese motivo, no calculamos MAE, RMSE ni R². La inferencia tiene como objetivo generar las estimaciones finales del modelo para cada pozo.

#### 20.1. Reentrenamiento con todos los objetivos conocidos

El modelo evaluado en test fue entrenado con objetivos hasta diciembre de 2025. Para predecir agosto de 2026, ese modelo no aprovecharía los siete meses más recientes, de enero a julio de 2026, que ya conocemos.

Una vez terminada la evaluación, reentrenamos la misma configuración (mismos hiperparámetros y mismo inicio de ventana, enero de 2020) agregando los objetivos de validación y test. Esta decisión no modifica las métricas informadas, que corresponden al modelo evaluado en test, y es la práctica habitual antes de usar un modelo en producción: la evaluación mide la configuración y el reentrenamiento final aprovecha toda la información disponible.

In [26]:
# Unimos los objetivos desde 2020 hasta julio de 2026.
entrenamiento_inferencia = pd.concat(
    [
        entrenamiento_final_regresion[features + [objetivo, "mes_objetivo"]],
        validation[features + [objetivo, "mes_objetivo"]],
        test[features + [objetivo, "mes_objetivo"]],
    ],
    ignore_index=True,
)

if entrenamiento_inferencia[objetivo].isna().any():
    raise ValueError("El entrenamiento para inferencia contiene objetivos faltantes.")

if entrenamiento_inferencia["mes_objetivo"].max() >= inferencia["mes_objetivo"].min():
    raise ValueError("El entrenamiento para inferencia incluye el mes que queremos predecir.")

print(
    "Objetivos de entrenamiento para inferencia:",
    entrenamiento_inferencia["mes_objetivo"].min().strftime("%Y-%m"),
    "a",
    entrenamiento_inferencia["mes_objetivo"].max().strftime("%Y-%m"),
)
print(f"Filas: {len(entrenamiento_inferencia):,}")

modelo_rf_inferencia = clone(modelo_rf_final)

inicio_reentrenamiento = time.time()
modelo_rf_inferencia.fit(
    entrenamiento_inferencia[features],
    entrenamiento_inferencia[objetivo],
)
print(f"Reentrenamiento finalizado en {time.time() - inicio_reentrenamiento:.1f} segundos.")

Objetivos de entrenamiento para inferencia: 2020-01 a 2026-07
Filas: 282,682


Reentrenamiento finalizado en 336.7 segundos.


#### 20.2. Generación de las predicciones

Utilizamos el Random Forest reentrenado con los objetivos comprendidos entre enero de 2020 y julio de 2026 para predecir la producción de agosto de 2026.

In [27]:
# Verificamos que el conjunto de inferencia tenga las variables
# predictoras necesarias para aplicar el modelo final.
columnas_faltantes_inferencia = sorted(
    set(features) - set(inferencia.columns)
)

if columnas_faltantes_inferencia:
    raise KeyError(
        "Faltan variables predictoras en el conjunto de inferencia: "
        f"{columnas_faltantes_inferencia}"
    )

# Generamos una predicción por cada registro del conjunto de inferencia.
predicciones_agosto_2026 = modelo_rf_inferencia.predict(
    inferencia[features]
)

if not np.isfinite(predicciones_agosto_2026).all():
    raise ValueError(
        "El modelo generó predicciones no finitas para la inferencia."
    )

if (predicciones_agosto_2026 < 0).any():
    raise ValueError(
        "El modelo generó predicciones negativas para la inferencia."
    )

resultado_inferencia = inferencia[
    ["idpozo", "sigla", "fecha", "mes_objetivo"]
].copy()

resultado_inferencia["produccion_predicha_m3"] = (
    predicciones_agosto_2026
)

print(
    f"Registros inferidos: {len(resultado_inferencia):,}"
)
print(
    "Mes de corte:",
    resultado_inferencia["fecha"].min().strftime("%Y-%m"),
    "a",
    resultado_inferencia["fecha"].max().strftime("%Y-%m"),
)
print(
    "Mes objetivo:",
    resultado_inferencia["mes_objetivo"].min().strftime("%Y-%m"),
    "a",
    resultado_inferencia["mes_objetivo"].max().strftime("%Y-%m"),
)

display(
    resultado_inferencia.head(10).style.format({
        "produccion_predicha_m3": "{:.2f}",
    })
)

Registros inferidos: 5,062
Mes de corte: 2026-07 a 2026-07
Mes objetivo: 2026-08 a 2026-08


,idpozo,sigla,fecha,mes_objetivo,produccion_predicha_m3
0,3640,PPC.Nq.EC-4,2026-07-01 00:00:00,2026-08-01 00:00:00,48.26
1,8043,YPF.Nq.FP.x-2,2026-07-01 00:00:00,2026-08-01 00:00:00,0.19
2,10073,CSJ.RN.LMa.a-3,2026-07-01 00:00:00,2026-08-01 00:00:00,21.03
3,11191,Q.SC.ACI-21,2026-07-01 00:00:00,2026-08-01 00:00:00,9.68
4,11641,TPT.Nq.PS-1001,2026-07-01 00:00:00,2026-08-01 00:00:00,3.10
5,11642,TPT.Nq.PS-1002,2026-07-01 00:00:00,2026-08-01 00:00:00,1.10
6,11643,YPF.Nq.PS.x-2,2026-07-01 00:00:00,2026-08-01 00:00:00,0.06
7,34833,PAE.Nq.LA.xp-89,2026-07-01 00:00:00,2026-08-01 00:00:00,0.10
8,42900,YPF.Nq.BP.x-2,2026-07-01 00:00:00,2026-08-01 00:00:00,0.00
9,42905,YPF.Nq.BP.a-7,2026-07-01 00:00:00,2026-08-01 00:00:00,0.40


#### 20.3. Resumen de las predicciones

Además de guardar la predicción individual de cada pozo, resumimos su distribución para comprobar que los valores generados sean razonables y detectar posibles resultados extremos.

Este resumen no reemplaza la evaluación con métricas. Como agosto de 2026 todavía no tiene un valor real disponible, solamente permite describir las predicciones producidas por el modelo.

In [28]:
resumen_inferencia = pd.DataFrame({
    "Cantidad de registros": [len(resultado_inferencia)],
    "Pozos": [resultado_inferencia["idpozo"].nunique()],
    "Predicción mínima (m³)": [
        resultado_inferencia["produccion_predicha_m3"].min()
    ],
    "Predicción media (m³)": [
        resultado_inferencia["produccion_predicha_m3"].mean()
    ],
    "Predicción mediana (m³)": [
        resultado_inferencia["produccion_predicha_m3"].median()
    ],
    "Percentil 90 (m³)": [
        resultado_inferencia[
            "produccion_predicha_m3"
        ].quantile(0.90)
    ],
    "Percentil 99 (m³)": [
        resultado_inferencia[
            "produccion_predicha_m3"
        ].quantile(0.99)
    ],
    "Predicción máxima (m³)": [
        resultado_inferencia["produccion_predicha_m3"].max()
    ],
    "Predicciones iguales a cero": [
        (
            resultado_inferencia["produccion_predicha_m3"] == 0
        ).sum()
    ],
})

display(resumen_inferencia.round(2))

,Cantidad de registros,Pozos,Predicción mínima (m³),Predicción media (m³),Predicción mediana (m³),Percentil 90 (m³),Percentil 99 (m³),Predicción máxima (m³),Predicciones iguales a cero
0,5062,5062,0.0,641.08,33.43,2007.97,6605.85,11312.76,342


#### 20.4. Exportación de las predicciones

Guardamos las predicciones en un archivo separado dentro de `data/processed`. El archivo conserva la identificación del pozo, el mes de corte, el mes objetivo y la producción estimada para agosto de 2026.

In [29]:
ruta_inferencia_final = (
    RUTA_PROCESSED / "predicciones_produccion_agosto_2026.csv"
)

resultado_inferencia.to_csv(
    ruta_inferencia_final,
    index=False,
)

print(f"Archivo guardado en: {ruta_inferencia_final}")
print(
    f"Filas guardadas: {len(resultado_inferencia):,}"
)

Archivo guardado en: /Users/julianvarelasilva/Downloads/ypf-data/data/processed/predicciones_produccion_agosto_2026.csv
Filas guardadas: 5,062


### 21. Conclusión del modelo de regresión

Con el análisis de los resultados generales y por grupo, damos por finalizada la etapa de regresión.

El modelo seleccionado es un Random Forest regresor entrenado con los objetivos comprendidos entre enero de 2020 y diciembre de 2025. Esta configuración utiliza 248.625 registros y 41 variables predictoras, manteniendo los hiperparámetros seleccionados durante la validación temporal.

En el conjunto de test, el modelo obtuvo:

- MAE: 133,616 m³;
- RMSE: 394,712 m³;
- R²: 0,908;
- predicciones negativas: 0.

La referencia de persistencia obtuvo un MAE de 166,195 m³, un RMSE de 551,710 m³ y un R² de 0,820. Por lo tanto, el Random Forest final redujo aproximadamente un 20 % el MAE y un 29 % el RMSE respecto de la referencia.

La comparación con el modelo entrenado solamente desde 2025 también favoreció a la ventana desde 2020. En el conjunto de test, el MAE disminuyó de 137,144 m³ a 133,616 m³ y el RMSE de 408,029 m³ a 394,712 m³.

El análisis por grupo mostró que el modelo funciona mejor en los registros con producción positiva, que representan el 70,91 % del conjunto de test. En este grupo, el MAE fue de 178,32 m³. Para los registros cuya producción real fue igual a cero, el MAE fue de 24,63 m³ y la predicción media fue positiva, lo que indica una ligera tendencia a sobreestimar algunos casos sin producción.

Los hiperparámetros no se fijaron manualmente: se seleccionaron con `GridSearchCV` sobre tres pliegues temporales construidos por mes del objetivo. En esos pliegues, el Random Forest obtuvo un MAE promedio de 131,39 m³ (desvío 3,50) frente a 163,87 m³ de la persistencia, y la mejora se mantuvo en los tres períodos. La misma búsqueda aplicada a Ridge eligió `alpha=1000`, pero Ridge siguió mostrando errores claramente mayores, lo que confirma que la relación entre las variables y la producción futura no es lineal.

Para la inferencia de agosto de 2026 reentrenamos la configuración final con todos los objetivos conocidos, desde enero de 2020 hasta julio de 2026 (282.682 registros), de modo que el modelo aproveche los meses más recientes. Las métricas informadas corresponden al modelo evaluado en test, entrenado hasta diciembre de 2025.

Estos resultados muestran que el modelo logra una mejora clara frente a una referencia sencilla y conserva un desempeño adecuado en meses posteriores a los utilizados para el entrenamiento. Sin embargo, no debe interpretarse como un predictor exacto para todos los pozos. Los errores son mayores en los casos de producción positiva y especialmente en los pozos con niveles de producción elevados.

A partir de este punto no se modifican nuevamente los hiperparámetros ni las variables predictoras del modelo de regresión.